# Đăng video lên GayVideo.fun (web + X) từ Google Photos

Notebook này lấy video từ **link chia sẻ Google Photos** `photos.app.goo.gl/…` (một video hoặc một album) rồi xử lý từng video. Link Google Drive chỉ là **phương án phụ**, dùng khi Google Photos chặn.

1. tải **file gốc**:
   - theo redirect của link rút gọn tới trang chia sẻ;
   - đọc các URL `lh3.googleusercontent.com` của video;
   - tải bản gốc bằng hậu tố `=dv`;
   - kiểm tra bằng **ffprobe** rằng file là video thật, không phải ảnh xem trước.
2. gắn watermark GayVideo.fun (giống luồng migration hiện tại);
3. upload lên **host chính**, và các host phụ nếu bật, để lấy mã file;
4. tạo thumbnail;
5. **caption AI** (nếu bật, cần GPU T4): JoyCaption xem 3–5 khung hình, mô hình chữ viết caption tiếng Anh kiểu nhá hàng theo giọng văn trong file caption mẫu của bạn, kèm gợi ý tag. Mô tả khung hình chỉ dùng nội bộ: không đăng, không gửi lên server. Mọi caption đều qua bộ lọc từ cấm và giới hạn ký tự của X;
6. **bạn duyệt** từng caption (sửa, hoặc bỏ chọn đăng X);
7. gọi **publish API**: lên web, và nếu bật thì vào hàng đợi đăng **X @Gayvideofun** qua Buffer, với clip ngắn cắt bằng ffmpeg (có logo và chữ `gayvideo.fun` ở cạnh dưới) và caption đã duyệt, kèm link bài viết.

### Chuẩn bị (một lần)
- **Colab Secrets** (biểu tượng 🔑 bên trái, bật *Notebook access* cho từng secret):
  - `GAYVIDEO_API_TOKEN`: token tạo trên VPS (`npm run api-token -- create …`);
  - `HOST_DOOD_API_KEY`, `HOST_VOE_API_KEY`, `HOST_EARNVIDS_API_KEY`: key của các host. Bắt buộc có key của host chính; key host phụ chỉ cần khi bật `UPLOAD_MIRRORS`;
  - chỉ khi dùng API để viết caption: `ANTHROPIC_API_KEY` hoặc `OPENAI_API_KEY`.
- **GPU cho caption AI:** *Runtime → Change runtime type → T4 GPU*. Lần đầu notebook tải JoyCaption (~17 GB) và Qwen3-8B (~16 GB) từ Hugging Face rồi nén 4-bit khi nạp, mất vài phút.
- **Caption mẫu:** `CAPTION_EXAMPLES` (mặc định `MyDrive/gayvideo/caption_examples.txt`), mỗi dòng một caption bạn chọn. Lần đầu notebook tự tạo file này từ mẫu khởi đầu để bạn sửa.
- **Link Google Photos** phải bật chia sẻ: *Chia sẻ → Tạo đường liên kết*.

### Cách dùng
1. Chạy ô **Cài đặt**.
2. Chỉnh ô **Cấu hình**.
3. Dán link vào ô **Danh sách video**, mỗi dòng một link, hoặc trỏ tới file CSV.
4. **Đánh dấu ô xác nhận.** Không xác nhận thì notebook không chạy.
5. Chạy ô **Chuẩn bị**: tải video, đề xuất caption (AI nếu bật).
6. Chạy ô **Duyệt caption**: sửa trực tiếp, hoặc bỏ chọn *Đăng X* (video vẫn lên web).
7. Chạy ô **Đăng**. Cuối cùng có bảng kết quả cho từng video.

### Cam kết của notebook
- **Không đăng trùng:** mỗi video đã đăng được ghi vào sổ `publish_log.csv` trên Google Drive (theo link và theo sha256 của file). Chạy lại thì các video đó được bỏ qua.
- **Lỗi một video không dừng cả lô:** lỗi được ghi vào bảng, rồi notebook làm tiếp video sau.
- **Dọn file tạm:** video tải về nằm trong `/content/gv-work` từ lúc Chuẩn bị tới lúc Đăng xong, rồi bị xoá.
- **Caption an toàn cho Buffer:** caption có từ cấm hoặc có link thì không được đăng X (ô Duyệt báo đỏ); caption quá dài bị cắt bớt theo cách X đếm ký tự.
- **Không lộ key:** key được đọc từ Colab Secrets và **không in ra**; thông báo lỗi luôn che key.

### Nếu Google Photos chặn
Google Photos không có API công khai để tải video từ link chia sẻ, và yt-dlp cũng không hỗ trợ (chỉ hỗ trợ Google Drive). Notebook dùng bộ đọc riêng trong `colab_helpers.py`:
- mỗi bước tự thử lại 3 lần;
- báo lỗi rõ cho từng link: chưa chia sẻ hoặc bị chặn đăng nhập, link đã xoá, album không có video, hoặc trang đã đổi giao diện;
- khi Google đổi giao diện, sửa `photos_media_urls()` và HTML mẫu trong `testdata/`, rồi chạy test (`python3 -m unittest discover -s scripts/publish-client`).

Nếu một link vẫn báo lỗi tải:
1. tải video đó lên **Google Drive**;
2. chia sẻ *Bất kỳ ai có đường liên kết*;
3. dán link Drive vào danh sách thay cho link Photos.

### Thư viện (tự sinh: các ô dưới ghi file client, helpers, caption AI và caption mẫu vào /content)

In [ ]:
%%writefile /content/gayvideo_publish.py
# Client publish API: tự sinh từ repo (build_notebook.py), đừng sửa ở đây.
"""Client for the GayVideo.fun publish API (docs/API.md).

Standard library only, so it runs as-is on Google Colab and on a laptop
(Python 3.8+). Two ways to use it:

    from gayvideo_publish import GayVideoClient
    client = GayVideoClient(token="gv_...")          # or env GAYVIDEO_API_TOKEN
    result = client.publish(
        title="Bevis Trần Đức Vương – My love story",
        mirrors={"dood": "abc123", "voe": "xyz789"},
        tags=["Bevis", "Couple"],
        category="couple",
        thumbnail="cover.jpg",                        # local file or https URL
        publish=False,                                # True = public right away
    )
    print(result["video"]["url"])

Also announce it on X (@Gayvideofun, through Buffer): pass an English
caption and the source video; a short clip (X_CLIP_SECONDS, from the
server) is cut here with ffmpeg, uploaded, and posted once the video is
public, with a link to its page:

    client.publish(..., publish=True,
                   x_caption="New: Bevis & Pinky's love story 🔥",
                   x_video="/content/full.mp4", x_clip_start=30)

    python gayvideo_publish.py --title "..." --mirror dood:abc123 --tag Bevis --publish \
        --x-caption "New video" --x-video full.mp4
"""

from __future__ import annotations

import argparse
import base64
import json
import mimetypes
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
from typing import Any, Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple, Union

DEFAULT_BASE_URL = "https://gayvideo.fun"
MAX_THUMBNAIL_BYTES = 8 * 1024 * 1024

Mirrors = Union[Mapping[str, str], Iterable[Tuple[str, str]]]


class PublishError(Exception):
    """The API refused the request. `status`, `code` and `body` say why."""

    def __init__(self, status: int, code: str, message: str, body: Optional[dict] = None):
        super().__init__(f"{status} {code}: {message}")
        self.status = status
        self.code = code
        self.body = body or {}


class _ProgressReader:
    """File wrapper that reports how many bytes urllib has read (sent) so far."""

    def __init__(self, handle: Any, progress: Callable[[int], None]):
        self.handle, self.progress, self.sent = handle, progress, 0

    def read(self, size: int = -1) -> bytes:
        chunk = self.handle.read(size)
        self.sent += len(chunk)
        self.progress(self.sent)
        return chunk


class GayVideoClient:
    def __init__(
        self,
        token: Optional[str] = None,
        base_url: Optional[str] = None,
        timeout: float = 120,
    ) -> None:
        self.token = (token or os.environ.get("GAYVIDEO_API_TOKEN") or "").strip()
        if not self.token.startswith("gv_"):
            raise ValueError("Missing token: pass token='gv_...' or set GAYVIDEO_API_TOKEN")
        self.base_url = (base_url or os.environ.get("GAYVIDEO_API_URL") or DEFAULT_BASE_URL).rstrip("/")
        self.timeout = timeout

    def _request(
        self,
        method: str,
        path: str,
        body: Optional[dict] = None,
        *,
        upload: Optional[str] = None,
        progress: Optional[Callable[[int], None]] = None,
    ) -> dict:
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Accept": "application/json",
            "User-Agent": "gayvideo-publish/2",
        }
        data: Any = None
        handle = None
        if upload is not None:
            # Streamed from disk, not read into memory.
            handle = open(upload, "rb")
            data = _ProgressReader(handle, progress) if progress else handle
            headers["Content-Type"] = "video/mp4"
            headers["Content-Length"] = str(os.path.getsize(upload))
        elif body is not None:
            data = json.dumps(body).encode("utf-8")
            headers["Content-Type"] = "application/json"
        request = urllib.request.Request(self.base_url + path, data=data, method=method, headers=headers)
        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as error:
            raw = error.read().decode("utf-8", "replace")
            try:
                payload = json.loads(raw)
            except ValueError:
                payload = {}
            info = payload.get("error", {}) if isinstance(payload, dict) else {}
            raise PublishError(
                error.code,
                info.get("code", "http_error"),
                info.get("message", raw[:200] or error.reason),
                payload if isinstance(payload, dict) else None,
            ) from None
        finally:
            if handle is not None:
                handle.close()

    def meta(self) -> dict:
        """Enabled hosts and category slugs; also checks the token works."""
        return self._request("GET", "/api/v1/meta")

    def get_video(self, video_id: str) -> dict:
        return self._request("GET", f"/api/v1/videos/{video_id}")["video"]

    def upload_clip(self, clip_path: str, progress: Optional[Callable[[int], None]] = None) -> dict:
        """Upload an MP4 made by cut_clip(); returns {"clipId", "bytes", "seconds"}.
        progress, if given, is called with the number of bytes sent so far."""
        return self._request("POST", "/api/v1/clips", upload=clip_path, progress=progress)

    def clip_seconds(self) -> int:
        """X_CLIP_SECONDS: from the environment if set, else the server's (0 = full video)."""
        local = os.environ.get("X_CLIP_SECONDS", "").strip()
        if local:
            return int(local)
        return int(self.meta().get("x", {}).get("clipSeconds", 120))

    def publish(
        self,
        title: str,
        mirrors: Mirrors,
        *,
        slug: Optional[str] = None,
        description: Optional[str] = None,
        category: Optional[str] = None,
        tags: Optional[Sequence[str]] = None,
        quality: Optional[str] = None,
        thumbnail: Optional[str] = None,
        publish: bool = False,
        x_caption: Optional[str] = None,
        x_video: Optional[str] = None,
        x_clip: Optional[str] = None,
        x_clip_id: Optional[str] = None,
        x_clip_start: float = 0,
        x_clip_seconds: Optional[int] = None,
        x_label: Optional[str] = "gayvideo.fun",
    ) -> dict:
        """Create the video, or update the draft its files already belong to.

        For X, pass x_caption (English) and either x_video (the source file;
        a clip is cut from x_clip_start, with x_label such as "gayvideo.fun"
        burned in at the bottom; None = no text), x_clip (an MP4 you cut yourself)
        or x_clip_id (a clip already sent with upload_clip()).
        The X post goes out once the video is public; it never blocks the web.

        Returns {"created": bool, "video": {...}, "x": {...}?, "warnings": [...]}.
        Raises PublishError on refusal (e.g. 409 already_published).
        """
        clip_id = x_clip_id
        if x_caption and not clip_id:
            if not (x_video or x_clip):
                raise ValueError("x_caption needs x_video (source to cut), x_clip (ready MP4) or x_clip_id")
            if x_clip:
                clip_id = self.upload_clip(x_clip)["clipId"]
            else:
                seconds = self.clip_seconds() if x_clip_seconds is None else x_clip_seconds
                workdir = tempfile.mkdtemp(prefix="gv-clip-")
                try:
                    clip_path = cut_clip(x_video, os.path.join(workdir, "clip.mp4"),
                                         start=x_clip_start, seconds=seconds, label=x_label)
                    clip_id = self.upload_clip(clip_path)["clipId"]
                finally:
                    shutil.rmtree(workdir, ignore_errors=True)

        pairs = list(mirrors.items()) if isinstance(mirrors, Mapping) else list(mirrors)
        body: Dict[str, Any] = {
            "title": title,
            "mirrors": [{"host": host, "fileCode": code} for host, code in pairs],
            "publish": publish,
        }
        if slug:
            body["slug"] = slug
        if description is not None:
            body["description"] = description
        if category:
            body["categorySlug"] = category
        if tags is not None:
            body["tags"] = list(tags)
        if quality:
            body["quality"] = quality
        if thumbnail:
            if thumbnail.startswith(("http://", "https://")):
                body["thumbnailUrl"] = thumbnail
            else:
                body["thumbnailDataUrl"] = image_data_url(thumbnail)
        if clip_id:
            body["x"] = {"caption": x_caption, "clipId": clip_id}
        return self._request("POST", "/api/v1/videos", body)


# Fonts tried for the clip label, in order; without one, ffmpeg's fontconfig
# picks a default sans font.
LABEL_FONTS = (
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/Library/Fonts/Arial Bold.ttf",
    "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
    "C:/Windows/Fonts/arialbd.ttf",
)


def _filter_value(value: str) -> str:
    """Escape a filter option value for both ffmpeg parsing levels: the
    option string (\\ ' :) and then the filter graph (\\ ' [ ] , ;)."""
    for level in ("\\':", "\\'[],;"):
        value = "".join("\\" + c if c in level else c for c in value)
    return value


def label_filter(label: str) -> str:
    """A drawtext filter that writes label at the bottom centre of the clip,
    white on a translucent box, sized to the frame (works for portrait too)."""
    font = next((f for f in LABEL_FONTS if os.path.exists(f)), None)
    font_opt = f"fontfile={_filter_value(font)}:" if font else "font=Sans:"
    return (
        f"drawtext={font_opt}expansion=none:text={_filter_value(label.strip())}:fontcolor=white:"
        "fontsize='max(16,min(w,h)/18)':box=1:boxcolor=black@0.45:boxborderw=10:"
        "x='(w-text_w)/2':y='h-text_h-min(w,h)/14'"
    )


def cut_clip(source: str, output: str, *, start: float = 0, seconds: int = 120,
             label: Optional[str] = None) -> str:
    """Cut and re-encode a clip X accepts: MP4, H.264 High (yuv420p), AAC-LC
    stereo 44.1 kHz, longest side at most 1280 and 60 fps, moov at the start
    (+faststart). seconds=0 keeps the whole video. label (e.g. "gayvideo.fun")
    is burned in at the bottom so the site stays visible if the clip is
    re-shared. Needs ffmpeg on PATH (Colab has it; Windows: winget install
    Gyan.FFmpeg; macOS: brew install ffmpeg).
    """
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg is not installed (see docs/API.md)")
    command = clip_command(source, output, start=start, seconds=seconds, label=label)
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(output):
        raise RuntimeError(f"ffmpeg failed: {result.stderr.strip()[-500:]}")
    return output


def clip_command(source: str, output: str, *, start: float = 0, seconds: int = 120,
                 label: Optional[str] = None) -> List[str]:
    """The ffmpeg command cut_clip() runs (callers may add -progress)."""
    command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"]
    if start:
        command += ["-ss", str(start)]
    command += ["-i", source]
    if seconds:
        command += ["-t", str(seconds)]
    # Longest side at most 1280 (720p landscape or portrait), even sizes.
    video_filter = "scale='if(gte(iw,ih),min(1280,iw),-2)':'if(gte(iw,ih),-2,min(1280,ih))'"
    if label and label.strip():
        video_filter += "," + label_filter(label)
    command += [
        "-map", "0:v:0", "-map", "0:a:0?",
        "-vf", video_filter,
        "-fpsmax", "60",
        "-c:v", "libx264", "-profile:v", "high", "-level:v", "4.1", "-pix_fmt", "yuv420p",
        "-preset", "medium", "-crf", "23", "-maxrate", "6M", "-bufsize", "12M",
        "-c:a", "aac", "-b:a", "128k", "-ar", "44100", "-ac", "2",
        "-movflags", "+faststart",
        output,
    ]
    return command


def image_data_url(path: str) -> str:
    size = os.path.getsize(path)
    if size > MAX_THUMBNAIL_BYTES:
        raise ValueError(f"{path} is {size} bytes; thumbnails are limited to 8 MB")
    mime = mimetypes.guess_type(path)[0] or ""
    if mime not in ("image/jpeg", "image/png", "image/webp", "image/gif"):
        raise ValueError(f"{path}: use a .jpg, .png, .webp or .gif image")
    with open(path, "rb") as handle:
        return f"data:{mime};base64," + base64.b64encode(handle.read()).decode("ascii")


def _parse_mirror(value: str) -> Tuple[str, str]:
    host, sep, code = value.partition(":")
    if not sep or not host or not code:
        raise argparse.ArgumentTypeError("use host:fileCode, e.g. dood:abc123")
    return host.strip(), code.strip()


def main(argv: Optional[List[str]] = None) -> int:
    parser = argparse.ArgumentParser(description="Publish a video to GayVideo.fun")
    parser.add_argument("--title", required=True)
    parser.add_argument("--mirror", action="append", type=_parse_mirror, required=True,
                        help="host:fileCode, repeat per host (dood, voe, earnvids)")
    parser.add_argument("--slug")
    parser.add_argument("--description")
    parser.add_argument("--category", help="category slug (see --meta)")
    parser.add_argument("--tag", action="append", dest="tags")
    parser.add_argument("--quality", choices=["HD", "4K"])
    parser.add_argument("--thumbnail", help="local image file or https URL")
    parser.add_argument("--publish", action="store_true", help="make it public now (default: draft)")
    parser.add_argument("--x-caption", help="also post on X with this English caption")
    parser.add_argument("--x-video", help="source video to cut the X clip from")
    parser.add_argument("--x-clip", help="a ready MP4 clip (skips cutting)")
    parser.add_argument("--x-start", type=float, default=0, help="clip start, in seconds")
    parser.add_argument("--x-seconds", type=int, default=None,
                        help="clip length (default: X_CLIP_SECONDS from the server; 0 = full)")
    parser.add_argument("--x-label", default="gayvideo.fun",
                        help='text burned into the bottom of the X clip ("" = none)')
    parser.add_argument("--base-url", default=None)
    args = parser.parse_args(argv)

    client = GayVideoClient(base_url=args.base_url)
    try:
        result = client.publish(
            args.title,
            args.mirror,
            slug=args.slug,
            description=args.description,
            category=args.category,
            tags=args.tags,
            quality=args.quality,
            thumbnail=args.thumbnail,
            publish=args.publish,
            x_caption=args.x_caption,
            x_video=args.x_video,
            x_clip=args.x_clip,
            x_clip_start=args.x_start,
            x_clip_seconds=args.x_seconds,
            x_label=args.x_label or None,
        )
    except PublishError as error:
        print(f"Refused: {error}", file=sys.stderr)
        return 1
    video = result["video"]
    state = "published" if video["published"] else "draft (approve it in /admin/review)"
    print(f"{'Created' if result['created'] else 'Updated'}: {client.base_url}{video['url']} [{state}]")
    if "x" in result:
        print(f"X: {result['x']['status']} (posted through Buffer once public)")
    for warning in result.get("warnings", []):
        print(f"warning: {warning}", file=sys.stderr)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/colab_helpers.py
# Helpers tải/upload: tự sinh từ repo (build_notebook.py), đừng sửa ở đây.
"""Helpers for colab_publish.ipynb: get a source video, upload it to the
hosts, and keep a ledger so nothing is posted twice.

Kept separate from the notebook so it can be tested (test_colab_helpers.py);
build_notebook.py embeds this file into the notebook. Needs requests (on
Colab by default), curl, ffmpeg/ffprobe, and for watermarks rsvg-convert
(apt librsvg2-bin) and Pillow.

Secrets (host API keys) are never printed: they are passed to curl in a
config file readable only by this user, and redact() strips them from every
error message.
"""

from __future__ import annotations

import csv
import datetime as dt
import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple
from urllib.parse import urlparse

import requests

BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
)
_SECRETS: List[str] = []


# --- secrets -----------------------------------------------------------------

def register_secret(value: Optional[str]) -> Optional[str]:
    """Remember a secret so redact() hides it; returns it unchanged."""
    if value and len(value) >= 4 and value not in _SECRETS:
        _SECRETS.append(value)
    return value


def redact(text: object, limit: int = 600) -> str:
    out = str(text)
    for secret in _SECRETS:
        out = out.replace(secret, "***")
    out = re.sub(r"(?i)(key|api_key|token)=([^&\s\"']+)", r"\1=***", out)
    return out.replace("\r", " ").replace("\n", " ").strip()[:limit]


# --- input --------------------------------------------------------------------

@dataclass
class Job:
    """One line of input: a share link plus optional metadata."""

    link: str
    title: str = ""
    tags: List[str] = field(default_factory=list)
    category: str = ""
    x_caption: str = ""
    clip_start: Optional[float] = None
    keywords: str = ""  # Vietnamese keywords for the AI caption (never posted)


def _split_tags(value: str) -> List[str]:
    return [tag.strip() for tag in re.split(r"[;,]", value or "") if tag.strip()]


def parse_lines(text: str) -> List[Job]:
    """`link | title | tags | category | x caption | clip start | keywords`,
    all but the link optional; blank lines and lines starting with # are ignored."""
    jobs = []
    for raw in (text or "").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = [part.strip() for part in line.split("|")]
        parts += [""] * (7 - len(parts))
        start = parts[5]
        jobs.append(
            Job(
                link=parts[0],
                title=parts[1],
                tags=_split_tags(parts[2]),
                category=parts[3],
                x_caption=parts[4],
                clip_start=float(start) if start else None,
                keywords="|".join(parts[6:]).strip(),
            )
        )
    return jobs


def parse_csv(path: str) -> List[Job]:
    """CSV with a header: link (required), title, tags, category, x_caption, clip_start, keywords."""
    jobs = []
    with open(path, newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            link = (row.get("link") or "").strip()
            if not link:
                continue
            start = (row.get("clip_start") or "").strip()
            jobs.append(
                Job(
                    link=link,
                    title=(row.get("title") or "").strip(),
                    tags=_split_tags(row.get("tags") or ""),
                    category=(row.get("category") or "").strip(),
                    x_caption=(row.get("x_caption") or "").strip(),
                    clip_start=float(start) if start else None,
                    keywords=(row.get("keywords") or "").strip(),
                )
            )
    return jobs


def ascii_text(value: str) -> str:
    """Accents stripped, đ -> d: for English captions built from Vietnamese titles."""
    folded = unicodedata.normalize("NFD", value)
    folded = "".join(ch for ch in folded if unicodedata.category(ch) != "Mn")
    return folded.replace("đ", "d").replace("Đ", "D")


# --- sources: Google Photos and Google Drive -------------------------------------

# --- progress -----------------------------------------------------------------------

def _clock(seconds: float) -> str:
    seconds = max(0, int(round(seconds)))
    return f"{seconds // 3600}:{seconds % 3600 // 60:02d}:{seconds % 60:02d}" if seconds >= 3600 \
        else f"{seconds // 60}:{seconds % 60:02d}"


def _mb(value: float) -> str:
    return f"{value / 1_048_576:.1f}"


class Step:
    """One step of one video, shown on a single line that is rewritten in
    place (carriage return), then replaced by "✓ name · 12.3 s" when done
    (or "✗ name · …" if it raised). unit "B" shows %, MB, MB/s and time
    left; unit "s" (ffmpeg) shows %, media time, speed (x real time) and
    time left; any other unit is a count ("2/4 khung hình"). total may be set later (e.g. from Content-Length).

        with Step("Tải Google Photos") as step:
            step.total = 1_000_000
            step.update(65_536)       # or step.set(absolute)
    """

    def __init__(self, name: str, total: Optional[float] = None, unit: str = "B", out: Any = None,
                 clock: Callable[[], float] = time.monotonic, interval: float = 0.5, indent: str = "  "):
        import sys

        self.name, self.total, self.unit, self.indent = name, total, unit, indent
        self.out = out or sys.stdout
        self.clock, self.interval = clock, interval
        self.done = 0.0
        self.started = self.last_draw = 0.0
        self.width = 0
        self.extra = ""

    def __enter__(self) -> "Step":
        self.started = self.clock()
        self.last_draw = self.started - self.interval
        self._draw(f"{self.name}…")
        return self

    def __exit__(self, kind: Any, error: Any, trace: Any) -> bool:
        elapsed = self.clock() - self.started
        summary = f"{'✗' if kind else '✓'} {self.name} · {elapsed:.1f} s"
        if not kind and self.unit == "B" and self.done:
            summary += f" · {_mb(self.done)} MB"
            if elapsed > 0:
                summary += f" · {_mb(self.done / elapsed)} MB/s"
        if self.extra:
            summary += f" · {self.extra}"
        self._draw(summary, final=True)
        return False

    def update(self, amount: float) -> None:
        self.set(self.done + amount)

    def set(self, value: float) -> None:
        self.done = max(0.0, float(value))
        now = self.clock()
        if now - self.last_draw >= self.interval:
            self.last_draw = now
            self._draw(self.line(now))

    def line(self, now: Optional[float] = None) -> str:
        elapsed = max(1e-6, (self.clock() if now is None else now) - self.started)
        rate = self.done / elapsed
        parts = [self.name]
        total = self.total if self.total and self.total > 0 else None
        if total:
            parts.append(f"{min(100.0, 100 * self.done / total):3.0f}%")
        if self.unit == "B":
            parts.append(f"{_mb(self.done)}/{_mb(total)} MB" if total else f"{_mb(self.done)} MB")
            parts.append(f"{_mb(rate)} MB/s")
        elif self.unit == "s":
            parts.append(f"{_clock(self.done)}/{_clock(total)}" if total else _clock(self.done))
            parts.append(f"{rate:.1f}x")
        else:  # a count, e.g. unit="khung hình"
            parts.append(f"{int(self.done)}/{int(total)} {self.unit}" if total else f"{int(self.done)} {self.unit}")
        if total and rate > 0 and self.done < total:
            parts.append(f"còn {_clock((total - self.done) / rate)}")
        return " · ".join(parts)

    def _draw(self, text: str, final: bool = False) -> None:
        text = self.indent + text
        pad = max(0, self.width - len(text))
        self.width = 0 if final else len(text)
        self.out.write("\r" + text + " " * pad + ("\n" if final else ""))
        self.out.flush()


def batch_line(index: int, count: int, title: str, out: Any = None) -> None:
    """The batch header, e.g. "▶ video 2/5 · My title"."""
    import sys

    (out or sys.stdout).write(f"▶ video {index}/{count} · {title}\n")
    (out or sys.stdout).flush()


class SourceError(RuntimeError):
    """The link could not be read; the message says what to do instead."""


@dataclass
class SourceItem:
    """One video found behind a link. `key` identifies it for the ledger."""

    key: str
    download: Callable[..., Path]  # download(target, step=None) -> target
    label: str = ""
    source: str = "Google Photos"


_PHOTOS_MEDIA = re.compile(r"https://lh3\.googleusercontent\.com/pw/[A-Za-z0-9_-]{20,}")
_DRIVE_FILE = re.compile(r"/file/d/([A-Za-z0-9_-]{10,})|[?&]id=([A-Za-z0-9_-]{10,})")
_DRIVE_FOLDER = re.compile(r"/folders/([A-Za-z0-9_-]{10,})")

PHOTOS_HELP = (
    "Hãy kiểm tra link Google Photos đã bật chia sẻ công khai (ai có link đều xem được). "
    "Nếu vẫn lỗi, Google Photos có thể đang chặn tải tự động: hãy tải video lên Google Drive, "
    "chia sẻ 'Bất kỳ ai có đường liên kết', rồi dùng link Drive thay thế."
)


def link_kind(link: str) -> str:
    host = urlparse(link).netloc.lower()
    if host in ("photos.app.goo.gl", "photos.google.com", "goo.gl"):
        return "photos"
    if host in ("drive.google.com", "docs.google.com"):
        return "drive"
    return "unknown"


def photos_media_urls(page: str) -> List[str]:
    """Distinct lh3 media base URLs on a shared Google Photos page (video,
    photo or album), in page order, without size suffixes."""
    seen, out = set(), []
    for url in _PHOTOS_MEDIA.findall(html.unescape(page).replace("\\u003d", "=").replace("\\/", "/")):
        if url not in seen:
            seen.add(url)
            out.append(url)
    return out


def photos_album_title(page: str) -> str:
    match = re.search(r'<meta property="og:title" content="([^"]*)"', page)
    title = html.unescape(match.group(1)).strip() if match else ""
    return "" if title.lower() in ("", "google photos", "photo", "video") else title


def _retry(action: Callable[[], Any], what: str, attempts: int = 3, base_delay: float = 3.0) -> Any:
    """Runs `action`, retrying network errors and _Retryable (429/5xx) with backoff."""
    last: Optional[BaseException] = None
    for attempt in range(attempts):
        try:
            return action()
        except (requests.RequestException, _Retryable) as error:
            last = error
            if attempt < attempts - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise SourceError(f"{what} thất bại sau {attempts} lần thử ({redact(last)}). {PHOTOS_HELP}")


class _Retryable(Exception):
    pass


def _photos_page(link: str, session: requests.Session) -> str:
    """Follows the short link (photos.app.goo.gl -> photos.google.com/share/…)
    and returns the share page, or explains why it can't."""

    def fetch() -> requests.Response:
        response = session.get(link, headers={"User-Agent": BROWSER_UA}, timeout=60, allow_redirects=True)
        if response.status_code == 429 or response.status_code >= 500:
            raise _Retryable(f"HTTP {response.status_code}")
        return response

    response = _retry(fetch, "Mở link Google Photos")
    final = urlparse(response.url)
    if final.netloc.startswith(("accounts.google.com", "consent.google.com")) or response.status_code in (401, 403):
        raise SourceError(f"Google Photos yêu cầu đăng nhập/đồng ý cookie (link chưa chia sẻ công khai hoặc bị chặn). {PHOTOS_HELP}")
    if response.status_code == 404:
        raise SourceError(f"Link Google Photos không tồn tại hoặc đã bị xoá (HTTP 404). {PHOTOS_HELP}")
    if response.status_code >= 400:
        raise SourceError(f"Google Photos trả HTTP {response.status_code}. {PHOTOS_HELP}")
    if final.netloc != "photos.google.com" or not final.path.startswith("/share/"):
        raise SourceError(f"Link không dẫn tới trang chia sẻ Google Photos (tới {final.netloc}{final.path[:40]}). {PHOTOS_HELP}")
    return response.text


IMAGE_CODECS = {"mjpeg", "png", "webp", "gif", "bmp", "tiff", "heif", "hevc_image", "av1_image"}


def verify_video(path: Path, min_seconds: float = 1.0) -> Dict[str, Any]:
    """ffprobe the download: it must hold a real video stream (not a still
    preview image) at least `min_seconds` long."""
    try:
        out = run(["ffprobe", "-v", "error", "-show_entries",
                   "stream=codec_type,codec_name,width,height:format=duration,format_name",
                   "-of", "json", str(path)], timeout=300)
        payload = json.loads(out)
    except Exception as error:
        raise SourceError(f"File tải về không đọc được bằng ffprobe ({redact(error, 200)}). {PHOTOS_HELP}") from None
    videos = [s for s in payload.get("streams", []) if s.get("codec_type") == "video"]
    duration = float(payload.get("format", {}).get("duration") or 0)
    if not videos or videos[0].get("codec_name") in IMAGE_CODECS or duration < min_seconds:
        kind = videos[0].get("codec_name") if videos else "không có luồng video"
        raise SourceError(f"File tải về là ảnh xem trước chứ không phải video ({kind}, {duration:.1f}s). {PHOTOS_HELP}")
    return {"duration": duration, "codec": videos[0].get("codec_name"),
            "width": videos[0].get("width"), "height": videos[0].get("height")}


def _download_photos_video(base: str, session: requests.Session) -> Callable[..., Path]:
    def download(target: Path, step: Optional[Step] = None) -> Path:
        # "=dv" asks Google for the original video file (undocumented, stable for years).
        def fetch() -> Path:
            if step:
                step.set(0)
            with session.get(base + "=dv", headers={"User-Agent": BROWSER_UA}, stream=True, timeout=(30, 600)) as response:
                if response.status_code == 429 or response.status_code >= 500:
                    raise _Retryable(f"HTTP {response.status_code}")
                kind = response.headers.get("content-type", "")
                if response.status_code >= 400 or not kind.startswith("video/"):
                    raise SourceError(
                        f"Google Photos không trả video gốc (HTTP {response.status_code}, {kind or 'không rõ loại'}). {PHOTOS_HELP}"
                    )
                length = response.headers.get("content-length", "")
                if response.headers.get("content-encoding", "identity") != "identity":
                    length = ""  # compressed: the length is not the file size
                if step and length.isdigit():
                    step.total = int(length)
                with open(target, "wb") as handle:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        handle.write(chunk)
                        if step:
                            step.update(len(chunk))
            if length.isdigit() and target.stat().st_size != int(length):
                raise _Retryable(f"tải thiếu ({target.stat().st_size}/{length} byte)")
            return target

        _retry(fetch, "Tải video gốc từ Google Photos")
        verify_video(target)
        return target

    return download


def _is_photos_video(base: str, session: requests.Session) -> bool:
    """Asks for one byte of "=dv": videos answer video/*, photos do not."""

    def fetch() -> bool:
        with session.get(base + "=dv", headers={"User-Agent": BROWSER_UA, "Range": "bytes=0-0"},
                         stream=True, timeout=60) as response:
            if response.status_code == 429 or response.status_code >= 500:
                raise _Retryable(f"HTTP {response.status_code}")
            return response.status_code < 400 and response.headers.get("content-type", "").startswith("video/")

    return bool(_retry(fetch, "Kiểm tra loại mục Google Photos"))


def photos_items(link: str, session: Optional[requests.Session] = None) -> tuple:
    """(album or item title, [SourceItem]) for every video behind a Photos link."""
    session = session or requests.Session()
    page = _photos_page(link, session)
    urls = photos_media_urls(page)
    if not urls:
        raise SourceError(
            "Không tìm thấy URL ảnh/video (lh3.googleusercontent.com/pw/…) trong trang chia sẻ: "
            "có thể Google đã đổi giao diện; cập nhật photos_media_urls() và HTML mẫu trong test. " + PHOTOS_HELP
        )
    videos = [url for url in urls if _is_photos_video(url, session)]
    if not videos:
        raise SourceError(f"Link Google Photos không có video nào tải được ({len(urls)} mục, đều là ảnh hoặc bị chặn). {PHOTOS_HELP}")
    items = [
        SourceItem(
            key="photos:" + hashlib.sha256(url.split("/pw/", 1)[1].encode()).hexdigest()[:24],
            download=_download_photos_video(url, session),
            label=f"{index + 1}/{len(videos)}",
        )
        for index, url in enumerate(videos)
    ]
    return photos_album_title(page), items


def drive_items(link: str) -> tuple:
    """A Drive file or folder shared as "anyone with the link" (uses gdown)."""
    try:
        import gdown  # noqa: F401  (pip install gdown; preinstalled on Colab)
    except ImportError:
        raise SourceError("Thiếu thư viện gdown: chạy `pip install gdown`.") from None
    import gdown

    folder = _DRIVE_FOLDER.search(link)
    if folder:
        workdir = Path(tempfile.mkdtemp(prefix="gv-drive-list-"))
        try:
            files = gdown.download_folder(url=link, output=str(workdir), quiet=True, skip_download=True)
        except Exception as error:  # gdown raises many types
            shutil.rmtree(workdir, ignore_errors=True)
            raise SourceError(f"Không đọc được thư mục Drive ({redact(error)}). Kiểm tra quyền 'Bất kỳ ai có đường liên kết'.") from None
        shutil.rmtree(workdir, ignore_errors=True)
        videos = [item for item in files or [] if re.search(r"\.(mp4|mov|m4v|mkv|webm|avi)$", item.path, re.I)]
        return "", [
            SourceItem(key=f"drive:{item.id}", download=_drive_download(item.id), label=Path(item.path).name,
                       source="Google Drive")
            for item in videos
        ]
    match = _DRIVE_FILE.search(link)
    if not match:
        raise SourceError("Link Google Drive không có mã file/thư mục.")
    file_id = match.group(1) or match.group(2)
    return "", [SourceItem(key=f"drive:{file_id}", download=_drive_download(file_id), source="Google Drive")]


def _drive_download(file_id: str) -> Callable[..., Path]:
    def download(target: Path, step: Optional[Step] = None) -> Path:
        import gdown

        if step:  # gdown draws its own one-line bar (%, MB, MB/s, time left)
            step.out.write("\n")
        result = gdown.download(id=file_id, output=str(target), quiet=step is None)
        if step and target.exists():
            step.set(target.stat().st_size)
        if not result or not target.exists() or target.stat().st_size < 10_000:
            raise SourceError("Tải file từ Google Drive thất bại (link chưa chia sẻ công khai, hoặc vượt hạn mức tải của Drive).")
        verify_video(target)
        return target

    return download


def source_items(link: str) -> tuple:
    kind = link_kind(link)
    if kind == "photos":
        return photos_items(link)
    if kind == "drive":
        return drive_items(link)
    raise SourceError("Chỉ hỗ trợ link Google Photos (photos.app.goo.gl/…) hoặc Google Drive.")


# --- video tools ------------------------------------------------------------------

def run(command: Sequence[str], timeout: int = 6 * 60 * 60) -> str:
    result = subprocess.run(list(command), capture_output=True, text=True, timeout=timeout)
    if result.returncode != 0:
        raise RuntimeError(f"{Path(command[0]).name} failed: {redact(result.stderr[-800:])}")
    return result.stdout


def run_ffmpeg(command: Sequence[str], duration: float, step: Optional[Step] = None,
               timeout: int = 6 * 60 * 60) -> None:
    """Runs an ffmpeg command with -progress pipe:1 and moves step to
    out_time / duration as it encodes. Raises RuntimeError on failure."""
    import threading

    args = [command[0], "-nostats", "-progress", "pipe:1", *command[1:]]
    if step:
        step.unit, step.total = "s", duration
    with tempfile.TemporaryFile("w+", encoding="utf-8", errors="replace") as errors:
        process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=errors, text=True)
        timer = threading.Timer(timeout, process.kill)
        timer.start()
        try:
            for line in process.stdout:  # key=value lines, a block every ~0.5 s
                key, _, value = line.strip().partition("=")
                if step and key in ("out_time_us", "out_time_ms") and value.lstrip("-").isdigit():
                    step.set(min(duration, max(0, int(value)) / 1_000_000))
                elif step and key == "progress" and value == "end":
                    step.set(duration)
            code = process.wait()
        finally:
            timer.cancel()
        if code != 0:
            errors.seek(0)
            raise RuntimeError(f"ffmpeg failed: {redact(errors.read()[-800:])}")


def sha256_file(path: Path, step: Optional[Step] = None) -> str:
    digest = hashlib.sha256()
    if step:
        step.total = path.stat().st_size
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
            if step:
                step.update(len(chunk))
    return digest.hexdigest()


def probe(path: Path) -> Dict[str, Any]:
    """{"duration": seconds, "width", "height"}; raises if there is no video stream."""
    out = run(["ffprobe", "-v", "error", "-select_streams", "v:0",
               "-show_entries", "stream=width,height:format=duration", "-of", "json", str(path)], timeout=300)
    payload = json.loads(out)
    streams = payload.get("streams") or []
    if not streams:
        raise RuntimeError("Không có luồng video trong file tải về.")
    return {
        "duration": float(payload.get("format", {}).get("duration") or 0),
        "width": int(streams[0]["width"]),
        "height": int(streams[0]["height"]),
    }


def make_thumbnail(source: Path, target: Path, duration: float, at: Optional[float] = None) -> Path:
    """A JPEG frame (default: 20% into the video, at least 1 s in)."""
    second = at if at is not None else max(1.0, duration * 0.2)
    second = min(second, max(0.0, duration - 0.5))
    run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", f"{second:.2f}", "-i", str(source),
         "-frames:v", "1", "-vf", "scale='min(1280,iw)':-2", "-q:v", "2", str(target)], timeout=300)
    return target


LOGO_URL = ("https://raw.githubusercontent.com/ronganminh/stream-video/main/"
            "scripts/publish-client/assets/logo-mark.svg")
PNG_MAGIC = b"\x89PNG\r\n\x1a\n"


class LogoError(RuntimeError):
    pass


def render_logo(logo: Path, size: int, out_png: Path) -> Path:
    """The logo as a size x size PNG: SVG through rsvg-convert, PNG through Pillow."""
    from PIL import Image

    if logo.suffix.lower() == ".png":
        image = Image.open(logo).convert("RGBA")
        image.thumbnail((size, size))
        image.save(out_png)
    else:
        run(["rsvg-convert", "-w", str(size), "-h", str(size), "-o", str(out_png), str(logo)], timeout=120)
    return out_png


def load_logo(url: str, folder: Path, session: Any = None, attempts: int = 3, base_delay: float = 3.0) -> Path:
    """Download the watermark logo and prove it renders, before any video is
    touched. url may also be a local path (e.g. a file on Drive). Raises LogoError with the reason (HTTP error, an HTML page
    instead of an image, invalid SVG/PNG, rsvg-convert failing)."""
    import xml.etree.ElementTree as ET

    data = b""
    if not url.startswith(("http://", "https://")):
        try:
            data = Path(url).expanduser().read_bytes()
        except OSError as error:
            raise LogoError(f"Không đọc được file logo {url}: {error}.") from None
        attempts = 0
    session = session or requests.Session()
    for attempt in range(attempts):
        try:
            response = session.get(url, timeout=60)
        except requests.RequestException as error:
            problem = f"lỗi mạng ({redact(str(error), 200)})"
        else:
            if response.status_code == 200:
                data = response.content
                break
            problem = f"HTTP {response.status_code}"
            if response.status_code != 429 and response.status_code < 500:
                raise LogoError(f"Không tải được logo {url}: {problem}.")
        if attempt == attempts - 1:
            raise LogoError(f"Không tải được logo {url} sau {attempts} lần thử: {problem}.")
        time.sleep(base_delay * (2 ** attempt))

    folder.mkdir(parents=True, exist_ok=True)
    if data.startswith(PNG_MAGIC):
        path = folder / "gv-logo.png"
    else:
        try:
            root = ET.fromstring(data)
        except ET.ParseError as error:
            head = data[:200].decode("utf-8", "replace").strip().lower()
            kind = "một trang HTML" if head.startswith(("<!doctype html", "<html")) else "không phải SVG/PNG"
            raise LogoError(f"logo {url} là {kind}, không dùng được ({error}).") from None
        tag = root.tag.rsplit("}", 1)[-1].lower()
        if tag == "html":
            raise LogoError(f"logo {url} là một trang HTML, không dùng được.")
        if tag != "svg":
            raise LogoError(f"logo {url}: XML nhưng không phải SVG (thẻ gốc <{tag}>).")
        path = folder / "gv-logo.svg"
    path.write_bytes(data)
    try:
        render_logo(path, 150, folder / "gv-logo-check.png")
    except Exception as error:
        raise LogoError(f"logo {url} không render được: {redact(str(error), 300)}") from None
    return path


def watermark(source: Path, target: Path, logo: Path, width: int, duration: float = 0,
              step: Optional[Step] = None) -> Path:
    """Same mark as the migration pipeline (scripts/video-migration/common.py):
    the logo with "GayVideo.fun" under it, top right, 82% opacity. logo is
    an SVG or PNG file (see load_logo)."""
    from PIL import Image, ImageDraw, ImageFont

    logo_size = max(48, min(150, round(width * 0.07)))
    font_size = max(12, min(28, round(width * 0.014)))
    margin = max(12, round(width * 0.0125))
    logo_png = render_logo(logo, logo_size, target.with_name("logo.png"))

    text_width = round(font_size * 7.4)
    canvas = Image.new("RGBA", (max(logo_size, text_width), logo_size + round(font_size * 1.65) + 4), (0, 0, 0, 0))
    logo = Image.open(logo_png).convert("RGBA")
    canvas.alpha_composite(logo, ((canvas.width - logo.width) // 2, 0))
    font = None
    for candidate in ("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"):
        if os.path.exists(candidate):
            font = ImageFont.truetype(candidate, font_size)
            break
    draw = ImageDraw.Draw(canvas)
    text = "GayVideo.fun"
    box = draw.textbbox((0, 0), text, font=font)
    draw.text(((canvas.width - (box[2] - box[0])) // 2, canvas.height - (box[3] - box[1]) - 4), text,
              font=font, fill=(255, 255, 255, 255), stroke_width=1, stroke_fill=(0, 0, 0, 153))
    alpha = canvas.getchannel("A").point(lambda value: round(value * 0.82))
    canvas.putalpha(alpha)
    mark = target.with_name("watermark.png")
    canvas.save(mark)

    run_ffmpeg(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source), "-i", str(mark),
                "-filter_complex", f"[1:v]format=rgba[wm];[0:v][wm]overlay=W-w-{margin}:{margin}[v]",
                "-map", "[v]", "-map", "0:a?", "-c:v", "libx264", "-preset", "veryfast", "-crf", "21",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "160k", "-movflags", "+faststart", str(target)],
               duration, step)
    return target


def cut_x_clip(source: Path, target: Path, duration: float, start: float = 0, seconds: int = 120,
               label: Optional[str] = None, step: Optional[Step] = None) -> Path:
    """The X clip, cut exactly as gayvideo_publish.cut_clip() does, with progress."""
    from gayvideo_publish import clip_command

    start = max(0.0, min(float(start or 0), max(0.0, duration - 1)))
    length = max(0.1, duration - start)
    if seconds:
        length = min(length, float(seconds))
    run_ffmpeg(clip_command(str(source), str(target), start=start, seconds=seconds, label=label), length, step)
    if not target.exists():
        raise RuntimeError("ffmpeg không tạo được clip X.")
    return target


# --- hosts --------------------------------------------------------------------------

HOSTS = {
    "dood": {"label": "DoodStream", "api": "https://doodapi.co/api", "secret": "HOST_DOOD_API_KEY"},
    "voe": {"label": "VOE", "api": "https://voe.sx/api", "secret": "HOST_VOE_API_KEY"},
    "earnvids": {"label": "EarnVids", "api": "https://earnvidsapi.com/api", "secret": "HOST_EARNVIDS_API_KEY"},
}


def _find_value(value: Any, keys: Iterable[str]) -> Any:
    keys = set(keys)
    if isinstance(value, dict):
        for key, item in value.items():
            if key in keys and item not in (None, ""):
                return item
        for item in value.values():
            found = _find_value(item, keys)
            if found is not None:
                return found
    if isinstance(value, list):
        for item in value:
            found = _find_value(item, keys)
            if found is not None:
                return found
    return None


def _first_url(value: Any) -> Optional[str]:
    if isinstance(value, str) and value.startswith(("http://", "https://")):
        return value
    if isinstance(value, dict):
        for item in value.values():
            found = _first_url(item)
            if found:
                return found
    if isinstance(value, list):
        for item in value:
            found = _first_url(item)
            if found:
                return found
    return None


_SIZE = re.compile(r"^(\d+(?:\.\d+)?)([kMGTP]?)$")


def _curl_size(value: str) -> Optional[float]:
    """curl meter sizes: "512", "45.2M", "1024k" (1024-based)."""
    match = _SIZE.match(value)
    if not match:
        return None
    return float(match.group(1)) * 1024 ** "_kMGTP".index(match.group(2) or "_")


def parse_curl_meter(line: str) -> Optional[Tuple[float, float]]:
    """(bytes sent, upload speed in bytes/s) from one line of curl's default
    progress meter, or None for header and other lines. Columns:
    %Total Total %Recv Received %Xferd Xferd AvgDload AvgUpload TimeTotal TimeSpent TimeLeft Speed"""
    parts = line.split()
    if len(parts) < 12 or not parts[0].isdigit():
        return None
    sent, speed = _curl_size(parts[5]), _curl_size(parts[7])
    if sent is None or speed is None:
        return None
    return sent, speed


def _run_curl(args: Sequence[str], step: Optional[Step]) -> Tuple[int, str, str]:
    """Runs curl, feeding its progress meter (stderr) to step.
    Returns (exit code, stdout, non-meter stderr lines)."""
    import threading

    env = {**os.environ, "LC_ALL": "C"}
    process = subprocess.Popen(list(args), stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env)
    out: List[bytes] = []
    reader = threading.Thread(target=lambda: out.append(process.stdout.read()), daemon=True)
    reader.start()
    messages: List[str] = []
    pending = b""
    while True:
        chunk = process.stderr.read1(4096) if hasattr(process.stderr, "read1") else process.stderr.read(4096)
        if not chunk:
            break
        pending += chunk
        *lines, pending = re.split(rb"[\r\n]", pending)
        for raw in lines:
            text = raw.decode("utf-8", "replace").strip()
            meter = parse_curl_meter(text)
            if meter:
                if step:
                    step.set(meter[0])
            elif text and not text.startswith(("% Total", "Dload")):
                messages.append(text)
    tail = pending.decode("utf-8", "replace").strip()
    if tail and not parse_curl_meter(tail):
        messages.append(tail)
    code = process.wait()
    reader.join()
    return code, b"".join(out).decode("utf-8", "replace"), "\n".join(messages[-5:])


class HostUploader:
    """Uploads like the migration pipeline does, without putting the key on a
    command line: curl reads the URL and form fields from a 0600 config file."""

    def __init__(self, host: str, key: str, session: Optional[requests.Session] = None):
        if host not in HOSTS:
            raise ValueError(f"Unknown host {host}")
        if not key:
            raise RuntimeError(f"Thiếu key {HOSTS[host]['secret']} trong Colab Secrets.")
        self.host, self.key, self.info = host, register_secret(key), HOSTS[host]
        self.session = session or requests.Session()

    def _get(self, path: str, **params: Any) -> Any:
        last = ""
        for attempt in range(5):
            try:
                response = self.session.get(self.info["api"] + path, params={"key": self.key, **params}, timeout=(20, 90))
                if response.status_code == 429 or response.status_code >= 500:
                    last = f"HTTP {response.status_code}"
                    time.sleep(2 ** attempt + 1)
                    continue
                if not response.ok:
                    raise RuntimeError(f"{self.info['label']} HTTP {response.status_code}: {redact(response.text, 200)}")
                return response.json()
            except (requests.RequestException, ValueError) as error:
                last = redact(error)
                time.sleep(2 ** attempt + 1)
        raise RuntimeError(f"{self.info['label']} API lỗi: {last}")

    def _curl(self, url: str, fields: Dict[str, str], path: Path, step: Optional[Step] = None) -> Any:
        if not shutil.which("curl"):
            raise RuntimeError("curl is not installed")

        def quote(value: str) -> str:
            return '"' + value.replace("\\", "\\\\").replace('"', '\\"') + '"'

        config = tempfile.NamedTemporaryFile("w", delete=False, prefix="gv-curl-", suffix=".cfg")
        try:
            os.chmod(config.name, 0o600)
            config.write(f"url = {quote(url)}\n")
            for name, value in fields.items():
                config.write(f"form = {quote(f'{name}={value}')}\n")
            config.write(f"form = {quote(f'file=@{path};filename={path.name};type=video/mp4')}\n")
            config.close()
            # No --silent: curl's progress meter goes to stderr (read below);
            # the response body goes to stdout. The key stays in the 0600 file.
            code, body, messages = _run_curl(
                ["curl", "--show-error", "--fail-with-body", "--connect-timeout", "60",
                 "--max-time", "21600", "--retry", "2", "--retry-delay", "10", "--retry-all-errors", "-K", config.name],
                step,
            )
        finally:
            os.unlink(config.name)
        if code != 0:
            raise RuntimeError(f"{self.info['label']} upload lỗi: {redact(messages or body, 300)}")
        try:
            return json.loads(body)
        except ValueError:
            raise RuntimeError(f"{self.info['label']} upload trả về không phải JSON: {redact(body, 200)}") from None

    def upload(self, path: Path, title: str, tags: Sequence[str] = (), step: Optional[Step] = None) -> str:
        """Uploads the file and returns the host's file code."""
        if step:
            step.unit, step.total = "B", path.stat().st_size
        server = _first_url(self._get("/upload/server"))
        if not server:
            raise RuntimeError(f"{self.info['label']} không trả về upload server.")
        if self.host == "dood":
            url = server if "?" in server else f"{server}?{self.key}"
            payload = self._curl(url, {"api_key": self.key}, path, step)
        elif self.host == "voe":
            payload = self._curl(f"{server}{'&' if '?' in server else '?'}key={self.key}", {}, path, step)
        else:
            payload = self._curl(server, {"key": self.key, "file_title": title, "tags": ",".join(tags),
                                          "file_public": "1", "file_adult": "1"}, path, step)
        code = _find_value(payload, ("filecode", "file_code", "fileCode", "code"))
        if not isinstance(code, str) or not code:
            raise RuntimeError(f"{self.info['label']} không trả về mã file: {redact(payload, 200)}")
        if title and self.host in ("dood", "voe"):
            try:
                self._get("/file/rename", file_code=code, title=title)
            except Exception:
                pass
        return code


# --- ledger -------------------------------------------------------------------------

LEDGER_FIELDS = ["time", "link", "item_key", "sha256", "title", "video_id", "url", "x", "status", "error"]


class Ledger:
    """A CSV of every item handled, so a link or file is never posted twice
    (kept on Google Drive so it survives Colab sessions)."""

    def __init__(self, path: str):
        self.path = path
        self.rows: List[Dict[str, str]] = []
        if os.path.exists(path):
            with open(path, newline="", encoding="utf-8") as handle:
                self.rows = list(csv.DictReader(handle))

    def done_keys(self) -> set:
        return {row["item_key"] for row in self.rows if row.get("status") == "done"}

    def done_hashes(self) -> set:
        return {row["sha256"] for row in self.rows if row.get("status") == "done" and row.get("sha256")}

    def add(self, **row: str) -> None:
        record = {name: str(row.get(name, "") or "") for name in LEDGER_FIELDS}
        record["time"] = record["time"] or dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        self.rows.append(record)
        os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
        new = not os.path.exists(self.path)
        with open(self.path, "a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=LEDGER_FIELDS)
            if new:
                writer.writeheader()
            writer.writerow(record)

In [ ]:
%%writefile /content/caption_ai.py
# Caption AI: tự sinh từ repo (build_notebook.py), đừng sửa ở đây.
"""AI captions for the X posts made by colab_publish.ipynb.

1. JoyCaption (a vision model, 4-bit on a Colab T4) describes 3-5 frames
   spread through the video. The description stays in memory: it is never
   posted, never written to the ledger and never sent to gayvideo.fun.
2. A text model writes one English teaser caption for X, plus a few tags
   for the web, from that description, the title, the tags, your
   Vietnamese keywords and the style of your example captions. By default
   an open model runs on the Colab GPU (no key); Anthropic or OpenAI can be
   used instead with a key in Colab Secrets (the description is then sent
   to that provider).
3. Every caption, AI-written or typed by you, goes through the same checks:
   no words from the banned list (Buffer forbids sexually explicit content)
   and at most CAPTION_BUDGET by X's weighted count, since the server adds
   a blank line and the link to the video page.

Everything that does not need a GPU is plain Python and covered by
test_caption_ai.py; the model classes import torch/transformers lazily.
"""

from __future__ import annotations

import gc
import json
import re
import subprocess
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

# --- X length (same rules as lib/social/xText.ts) -----------------------------------

X_MAX_WEIGHT = 280
URL_WEIGHT = 23
# The server posts "caption\n\nhttps://gayvideo.fun/watch/…": the URL weighs 23.
CAPTION_BUDGET = X_MAX_WEIGHT - URL_WEIGHT - 2


def _char_weight(char: str) -> int:
    code = ord(char)
    if (code <= 0x10FF or 0x2000 <= code <= 0x200D or 0x2010 <= code <= 0x201F
            or 0x2032 <= code <= 0x2037):
        return 1
    return 2


def x_weight(text: str) -> int:
    """X's weighted length (twitter-text v3) of plain text without URLs."""
    return sum(_char_weight(char) for char in unicodedata.normalize("NFC", text))


def fit_caption(caption: str, budget: int = CAPTION_BUDGET) -> str:
    """Collapses spaces and trims to `budget` with "…", as the server would."""
    clean = re.sub(r"[ \t]+", " ", unicodedata.normalize("NFC", caption or "")).strip()
    if x_weight(clean) <= budget:
        return clean
    room = budget - x_weight("…")
    out, weight = "", 0
    for char in clean:
        if weight + _char_weight(char) > room:
            break
        out += char
        weight += _char_weight(char)
    # Cut at a word boundary when one is close, so no half word is left.
    if " " in out[-25:]:
        out = out[: out.rfind(" ")]
    return out.rstrip(" ,;:-") + "…"


# --- banned words -------------------------------------------------------------------

# Buffer's terms forbid sexually explicit content: captions must tease, not
# describe. Matching ignores case, accents and separators ("f.u.c.k",
# "s3x"); add your own with BANNED_EXTRA in the notebook.
BANNED_WORDS = (
    "anal", "anus", "bareback", "bdsm", "blowjob", "blow job", "bj", "boner", "bukkake", "butthole",
    "cock", "cocks", "creampie", "cum", "cumming", "cumshot", "deepthroat", "dick", "dicks", "dildo",
    "ejaculate", "ejaculation", "erection", "fap", "fellatio", "fisting", "fuck", "fucked", "fucking",
    "fucks", "gangbang", "handjob", "hardcore", "horny", "jerk off", "jerking", "jizz", "masturbate",
    "masturbating", "milf", "naked", "nude", "nudes", "nudity", "orgasm", "orgy", "penis", "porn",
    "porno", "pornhub", "pussy", "rimjob", "rimming", "semen", "sex", "sexual", "slut",
    "sperm", "suck", "sucking", "testicles", "threesome", "twink", "xxx", "onlyfans", "nsfw",
    # Never acceptable in any caption.
    "teen", "teens", "underage", "minor", "schoolboy", "barely legal", "young boy", "boy", "boys",
    "lolita", "rape", "forced", "drugged", "incest", "stepbro", "stepdad", "stepson",
    # Vietnamese, in case keywords slip into a caption.
    "địt", "đụ", "chịch", "lồn", "cặc", "buồi", "bú cu", "thủ dâm", "khiêu dâm",
)

_LEET = str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t", "@": "a", "$": "s", "!": "i"})


def _fold(text: str) -> str:
    """Lower case, no accents (except đ→d), leetspeak undone."""
    text = unicodedata.normalize("NFD", (text or "").lower().replace("đ", "d"))
    text = "".join(char for char in text if unicodedata.category(char) != "Mn")
    return text.translate(_LEET)


def _tokens(text: str) -> List[str]:
    """Folded words; runs of 3+ single letters are joined ("f.u.c.k" → "fuck")."""
    out: List[str] = []
    run: List[str] = []
    for word in re.findall(r"[a-z]+", _fold(text)) + [""]:
        if len(word) == 1:
            run.append(word)
            continue
        if len(run) >= 3:
            out.append("".join(run))
        else:
            out.extend(run)
        run = []
        if word:
            out.append(word)
    return out


def banned_terms(text: str, extra: Iterable[str] = ()) -> List[str]:
    """The banned words found in text (empty list = clean). Whole words only,
    plus a plural "s", so "Sussex" or "cocktail" are fine."""
    joined = " " + " ".join(_tokens(text)) + " "
    found = []
    for term in list(BANNED_WORDS) + [t for t in extra if t and t.strip()]:
        folded = " ".join(re.findall(r"[a-z]+", _fold(term)))
        if not folded or term in found:
            continue
        if f" {folded} " in joined or (" " not in folded and f" {folded}s " in joined):
            found.append(term)
    return found


# --- captions -----------------------------------------------------------------------

GENERIC_CAPTION = "New video just dropped 🔥"


def ascii_title(value: str) -> str:
    value = unicodedata.normalize("NFD", (value or "").replace("đ", "d").replace("Đ", "D"))
    return "".join(char for char in value if unicodedata.category(char) != "Mn")


def default_caption(template: str, title: str, extra_banned: Iterable[str] = ()) -> str:
    """The template caption ("New video: {title} 🔥"); generic if the title
    has a banned word or the template is broken."""
    try:
        caption = fit_caption((template or "").format(title=ascii_title(title)))
    except (KeyError, IndexError, ValueError):
        caption = ""
    if not caption or banned_terms(caption, extra_banned):
        return GENERIC_CAPTION
    return caption


def clean_tags(tags: Iterable[str], extra_banned: Iterable[str] = (), limit: int = 8) -> List[str]:
    """Deduped (case-insensitive), no '#', short, not banned, at most `limit`."""
    out: List[str] = []
    seen = set()
    for tag in tags or ():
        tag = re.sub(r"\s+", " ", str(tag).replace("#", "")).strip(" ,;.")
        if not tag or len(tag) > 30 or banned_terms(tag, extra_banned):
            continue
        key = tag.casefold()
        if key not in seen:
            seen.add(key)
            out.append(tag)
        if len(out) >= limit:
            break
    return out


def parse_tags(text: str) -> List[str]:
    return [tag.strip() for tag in re.split(r"[;,]", text or "") if tag.strip()]


# --- example captions ----------------------------------------------------------------

def load_examples(path: str, starter: Optional[str] = None, extra_banned: Iterable[str] = ()) -> Tuple[List[str], List[str]]:
    """(examples, warnings) from a text file, one caption per line; lines
    starting with # are comments. A missing file is created from `starter`
    (the repo's assets/caption_examples.txt) so it can be edited on Drive."""
    file = Path(path).expanduser()
    warnings: List[str] = []
    if not file.exists() and starter is not None:
        try:
            file.parent.mkdir(parents=True, exist_ok=True)
            file.write_text(starter, encoding="utf-8")
            warnings.append(f"Đã tạo file mẫu {file} (sửa thoải mái, mỗi dòng 1 caption).")
        except OSError as error:
            warnings.append(f"Không tạo được {file} ({error}); dùng mẫu khởi đầu.")
            text = starter
        else:
            text = file.read_text(encoding="utf-8")
    elif file.exists():
        text = file.read_text(encoding="utf-8")
    else:
        text = ""
    examples = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        hits = banned_terms(line, extra_banned)
        if hits:
            warnings.append(f"Bỏ caption mẫu có từ cấm ({', '.join(hits)}): {line[:60]}")
            continue
        examples.append(line)
    return examples, warnings


# --- prompts ------------------------------------------------------------------------

CAPTION_SCHEMA = {
    "type": "object",
    "properties": {
        "caption": {"type": "string"},
        "tags": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["caption", "tags"],
    "additionalProperties": False,
}


def build_prompts(description: str, title: str, tags: Sequence[str], keywords: str,
                  examples: Sequence[str], budget: int = CAPTION_BUDGET,
                  feedback: str = "") -> Tuple[str, str]:
    """(system, user) for the text model."""
    system = (
        "You write the text of X (Twitter) posts that announce a new video on GayVideo.fun, "
        "an 18+ website for gay men. The post goes out through Buffer, whose rules forbid sexually "
        "explicit content, so every caption is a teaser:\n"
        "- playful, confident, a little flirty at most; build curiosity about the video;\n"
        "- never describe sex acts, nudity or genitals, never use explicit or vulgar words, "
        "never mention age, school, family roles or anything that suggests a minor;\n"
        "- English only (translate the meaning of Vietnamese keywords, do not copy them);\n"
        f"- at most {budget} characters, one or two short sentences, at most 2 emoji and 2 hashtags;\n"
        "- no URL and no \"link in bio\": the link to the video is added after the caption;\n"
        "- match the voice, rhythm and length of the example captions.\n"
        "Also suggest 3 to 6 short, non-explicit English tags for the website "
        "(people, setting, mood or style; no '#').\n"
        'Reply with JSON only: {"caption": "...", "tags": ["...", "..."]}'
    )
    parts = [f"Video title: {title or '(none)'}"]
    if tags:
        parts.append("Existing tags: " + ", ".join(tags))
    if keywords:
        parts.append("My keywords (Vietnamese, for meaning only): " + keywords)
    if description:
        parts.append(
            "What the frames show (private notes, may be explicit; use them only for the people, "
            "setting and mood, never repeat explicit details):\n" + description
        )
    if examples:
        parts.append("Example captions in my style:\n" + "\n".join(f"- {line}" for line in examples[:30]))
    if feedback:
        parts.append("Your previous answer was rejected: " + feedback + " Write a new one.")
    return system, "\n\n".join(parts)


def parse_writer_output(text: str) -> Tuple[str, List[str]]:
    """(caption, tags) from a model reply: JSON, possibly inside code fences
    or after a <think> block. Raises ValueError when there is no caption."""
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    data: Any = None
    for match in re.finditer(r"\{.*\}", text, flags=re.S):
        try:
            data = json.loads(match.group(0))
            break
        except ValueError:
            continue
    if not isinstance(data, dict):
        raise ValueError("model reply is not JSON")
    caption = str(data.get("caption") or "").strip().strip('"').strip()
    tags = data.get("tags") or []
    if isinstance(tags, str):
        tags = parse_tags(tags)
    if not caption:
        raise ValueError("model reply has no caption")
    return caption, [str(tag) for tag in tags if str(tag).strip()]


def check_caption(caption: str, extra_banned: Iterable[str] = (), budget: int = CAPTION_BUDGET) -> Tuple[str, List[str]]:
    """(caption fitted to the budget, problems). Problems name banned words or
    a URL; an over-long caption is trimmed, not refused."""
    problems = []
    hits = banned_terms(caption, extra_banned)
    if hits:
        problems.append("có từ cấm: " + ", ".join(hits))
    if re.search(r"https?://|www\.|\.(com|fun|net)\b", caption or "", flags=re.I):
        problems.append("có link (server tự thêm link bài viết)")
    return fit_caption(caption, budget), problems


@dataclass
class Suggestion:
    caption: str
    tags: List[str] = field(default_factory=list)
    source: str = "mặc định"  # "LINKS", "AI" or "mặc định"
    note: str = ""


def write_caption(generate: Callable[[str, str], str], *, description: str, title: str, tags: Sequence[str],
                  keywords: str, examples: Sequence[str], extra_banned: Iterable[str] = (),
                  attempts: int = 3, budget: int = CAPTION_BUDGET) -> Suggestion:
    """Asks `generate(system, user)` for a caption until one passes the checks.
    Raises RuntimeError after `attempts` rejected or failed replies."""
    extra_banned = list(extra_banned)
    feedback, last = "", ""
    for attempt in range(attempts):
        system, user = build_prompts(description, title, tags, keywords, examples, budget, feedback)
        try:
            caption, suggested = parse_writer_output(generate(system, user))
        except WriterRefused as error:
            raise RuntimeError(f"mô hình từ chối viết ({error})") from None
        except ValueError as error:
            feedback, last = f"{error}. Reply with the JSON object only.", str(error)
            continue
        fitted, problems = check_caption(caption, extra_banned, budget)
        if problems:
            feedback = "; ".join(problems) + ". Keep it a teaser with no explicit words."
            last = "; ".join(problems)
            continue
        if x_weight(caption) > budget:
            feedback = f"it is longer than {budget} characters."
            last = "quá dài"
            if attempt < attempts - 1:
                continue
        return Suggestion(fitted, clean_tags(suggested, extra_banned), "AI")
    raise RuntimeError(f"không có caption đạt yêu cầu sau {attempts} lần ({last})")


# --- review -------------------------------------------------------------------------

@dataclass
class Decision:
    post_x: bool
    caption: str
    tags: List[str]
    problems: List[str]


def review_decision(post_x: bool, caption: str, tags_text: str, extra_banned: Iterable[str] = (),
                    budget: int = CAPTION_BUDGET) -> Decision:
    """What the Đăng cell does with one reviewed row: a caption with problems
    (banned words, a link) or an empty one is not posted to X."""
    tags = clean_tags(parse_tags(tags_text), extra_banned, limit=20)
    caption = (caption or "").strip()
    if not post_x:
        return Decision(False, caption, tags, [])
    if not caption:
        return Decision(False, "", tags, ["caption trống"])
    fitted, problems = check_caption(caption, extra_banned, budget)
    return Decision(not problems, fitted, tags, problems)


def weight_label(caption: str, budget: int = CAPTION_BUDGET) -> str:
    weight = x_weight((caption or "").strip())
    return f"{weight}/{budget}" + (" (sẽ bị cắt bớt)" if weight > budget else "")


# --- frames -------------------------------------------------------------------------

def frame_times(duration: float, count: int) -> List[float]:
    """`count` (3-5) timestamps spread evenly, away from the very start/end."""
    count = max(1, min(5, int(count)))
    if duration <= 0:
        return [0.0]
    return [round(duration * (index + 0.5) / count, 2) for index in range(count)]


def extract_frames(video: Path, folder: Path, duration: float, count: int = 4, width: int = 768) -> List[Path]:
    """JPEG frames at frame_times(); the longer side is at most `width`."""
    folder.mkdir(parents=True, exist_ok=True)
    frames = []
    for index, second in enumerate(frame_times(duration, count)):
        target = folder / f"frame-{index + 1}.jpg"
        result = subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", f"{second:.2f}", "-i", str(video),
             "-frames:v", "1", "-vf", f"scale='if(gte(iw,ih),min({width},iw),-2)':'if(gte(iw,ih),-2,min({width},ih))'",
             "-q:v", "3", str(target)],
            capture_output=True, text=True, timeout=300,
        )
        if result.returncode == 0 and target.exists() and target.stat().st_size > 0:
            frames.append(target)
    if not frames:
        raise RuntimeError("ffmpeg không lấy được khung hình nào.")
    return frames


def merge_descriptions(descriptions: Sequence[Tuple[float, str]]) -> str:
    return "\n".join(f"Frame {index + 1} at {int(second) // 60}:{int(second) % 60:02d}: {text.strip()}"
                     for index, (second, text) in enumerate(descriptions) if text and text.strip())


# --- models (GPU, lazy imports) -----------------------------------------------------

JOYCAPTION_MODEL = "fancyfeast/llama-joycaption-beta-one-hf-llava"
LOCAL_TEXT_MODEL = "Qwen/Qwen3-8B"
ANTHROPIC_MODEL = "claude-opus-5-5"
OPENAI_MODEL = "gpt-5-mini"
JOYCAPTION_PROMPT = (
    "Write a descriptive caption for this image in a casual tone, under 90 words. "
    "Describe the people (build, hair, clothing), the setting, the lighting and the mood."
)


class WriterRefused(Exception):
    """The text model or API declined the request."""


def gpu_status() -> Tuple[bool, str]:
    """(has a CUDA GPU, human-readable name or reason)."""
    try:
        import torch
    except ImportError:
        return False, "chưa cài torch"
    if not torch.cuda.is_available():
        return False, "runtime không có GPU (Runtime → Change runtime type → T4 GPU)"
    return True, torch.cuda.get_device_name(0)


def _bnb_4bit() -> Any:
    import torch
    from transformers import BitsAndBytesConfig

    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=torch.float16,
                              # The vision tower and projector stay in fp16: quantizing them hurts captions.
                              llm_int8_skip_modules=["vision_tower", "multi_modal_projector", "lm_head"])


def free_gpu(*objects: Any) -> None:
    for obj in objects:
        del obj
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


class JoyCaptioner:
    """JoyCaption (LLaVA) in 4-bit NF4 with fp16 compute, which a T4 supports
    (bf16 is not native there). About 6 GB of VRAM."""

    def __init__(self, model_id: str = JOYCAPTION_MODEL):
        import torch
        from transformers import AutoProcessor, LlavaForConditionalGeneration

        self.torch = torch
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            model_id, quantization_config=_bnb_4bit(), torch_dtype=torch.float16, device_map=0)
        self.model.eval()

    def describe(self, image_path: Path, prompt: str = JOYCAPTION_PROMPT) -> str:
        from PIL import Image

        torch = self.torch
        convo = [{"role": "system", "content": "You are a helpful image captioner."},
                 {"role": "user", "content": prompt}]
        # As in the model card: this exact apply_chat_template + processor pairing.
        text = self.processor.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)
        with Image.open(image_path) as image:
            inputs = self.processor(text=[text], images=[image.convert("RGB")], return_tensors="pt").to("cuda")
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
        with torch.no_grad():
            ids = self.model.generate(**inputs, max_new_tokens=200, do_sample=True, suppress_tokens=None,
                                      use_cache=True, temperature=0.6, top_k=None, top_p=0.9)[0]
        ids = ids[inputs["input_ids"].shape[1]:]
        return self.processor.tokenizer.decode(ids, skip_special_tokens=True,
                                               clean_up_tokenization_spaces=False).strip()

    def close(self) -> None:
        model, processor = self.model, self.processor
        self.model = self.processor = None
        free_gpu(model, processor)


class LocalWriter:
    """An open instruct model in 4-bit on the Colab GPU (default Qwen3-8B,
    Apache-2.0, no Hugging Face token needed; thinking mode off)."""

    def __init__(self, model_id: str = LOCAL_TEXT_MODEL):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        config = _bnb_4bit()
        config.llm_int8_skip_modules = None
        self.model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=config,
                                                          torch_dtype=torch.float16, device_map=0)
        self.model.eval()

    def __call__(self, system: str, user: str) -> str:
        torch = self.torch
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        try:
            text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                      enable_thinking=False)
        except TypeError:
            text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to("cuda")
        with torch.no_grad():
            ids = self.model.generate(**inputs, max_new_tokens=300, do_sample=True, temperature=0.8, top_p=0.9)[0]
        return self.tokenizer.decode(ids[inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def close(self) -> None:
        model, tokenizer = self.model, self.tokenizer
        self.model = self.tokenizer = None
        free_gpu(model, tokenizer)


class AnthropicWriter:
    """Claude through the Anthropic SDK (key: Colab secret ANTHROPIC_API_KEY).
    Structured output guarantees JSON; server-side fallbacks retry a
    declined request on another model before reporting a refusal."""

    def __init__(self, api_key: str, model: str = ANTHROPIC_MODEL):
        import anthropic

        self.anthropic = anthropic
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model

    def __call__(self, system: str, user: str) -> str:
        anthropic = self.anthropic
        try:
            response = self.client.beta.messages.create(
                model=self.model,
                max_tokens=8000,
                betas=["server-side-fallback-2026-07-01"],
                fallbacks="default",
                output_config={"effort": "low", "format": {"type": "json_schema", "schema": CAPTION_SCHEMA}},
                system=system,
                messages=[{"role": "user", "content": user}],
            )
        except anthropic.RateLimitError as error:
            raise RuntimeError(f"Anthropic: quá giới hạn tốc độ ({error.status_code})") from None
        except anthropic.APIStatusError as error:
            raise RuntimeError(f"Anthropic: HTTP {error.status_code}") from None
        except anthropic.APIConnectionError:
            raise RuntimeError("Anthropic: lỗi kết nối") from None
        if response.stop_reason == "refusal":
            category = getattr(getattr(response, "stop_details", None), "category", None)
            raise WriterRefused(f"refusal{f' ({category})' if category else ''}")
        return next((block.text for block in response.content if block.type == "text"), "")

    def close(self) -> None:
        self.client = None


class OpenAIWriter:
    """OpenAI chat completions in JSON mode (key: Colab secret OPENAI_API_KEY)."""

    def __init__(self, api_key: str, model: str = OPENAI_MODEL):
        from openai import OpenAI

        self.client = OpenAI(api_key=api_key)
        self.model = model

    def __call__(self, system: str, user: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            response_format={"type": "json_object"},
        )
        choice = response.choices[0]
        if getattr(choice, "finish_reason", "") == "content_filter" or getattr(choice.message, "refusal", None):
            raise WriterRefused(str(getattr(choice.message, "refusal", "") or "content_filter"))
        return choice.message.content or ""

    def close(self) -> None:
        self.client = None


def ensure_package(module: str, package: Optional[str] = None) -> None:
    """pip-installs `package` if `module` cannot be imported (Colab)."""
    import importlib
    import sys

    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package or module], check=True)
        importlib.invalidate_caches()

In [ ]:
%%writefile /content/caption_examples.txt
# Caption mẫu khởi đầu: tự sinh từ repo (build_notebook.py), đừng sửa ở đây.
# Caption mẫu cho X: mỗi dòng 1 caption tiếng Anh, kiểu nhá hàng, không từ ngữ tình dục.
# AI bắt chước giọng văn, độ dài và cách dùng emoji/hashtag của các dòng này.
# Dòng bắt đầu bằng # là ghi chú. Sửa, thêm, xoá thoải mái.
Some mornings are better than others. This was one of them ☀️
He said "just one more rep"… we stopped counting 💪
Saturday plans: cancelled. Reason: this 👀
Two guys, one quiet weekend, zero chill 🔥
You were not ready for how this ends. Neither were we.
Gym crush, finally on camera. Worth the wait? You tell us 😏
Warm lights, cold drinks, and a story that heats up fast 🌙
New face, big energy. Remember the name.
The beach was empty. They didn't mind 🌊
Long day, longer night. Full video is up 👇
Some chemistry you just can't fake 🧪🔥
Started as a photoshoot. Didn't stay one 📸

In [ ]:
#@title 1. Cài đặt (chạy một lần mỗi phiên)
!apt-get -qq install -y librsvg2-bin > /dev/null
!pip -q install gdown requests pillow
!pip -q install -U bitsandbytes accelerate  # caption AI (4-bit trên GPU T4)
import importlib, sys
sys.path.insert(0, "/content")
for name in ("colab_helpers", "gayvideo_publish", "caption_ai"):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
print("OK: ffmpeg", "có" if __import__("shutil").which("ffmpeg") else "THIẾU")

In [ ]:
#@title 2. Cấu hình
API_URL = "https://gayvideo.fun"  #@param {type:"string"}
#@markdown Lên web ngay (bỏ chọn = bản nháp để duyệt trong /admin/review; bài X chờ tới khi duyệt).
PUBLISH = True  #@param {type:"boolean"}
#@markdown Đăng thêm lên X @Gayvideofun qua Buffer (clip ngắn + link). Độ dài clip theo `X_CLIP_SECONDS` trên server.
POST_TO_X = True  #@param {type:"boolean"}
#@markdown Caption X mặc định (tiếng Anh) khi dòng không có caption riêng; `{title}` = tiêu đề bỏ dấu.
X_CAPTION_TEMPLATE = "New video: {title} 🔥"  #@param {type:"string"}
#@markdown Clip X bắt đầu từ giây thứ (khi dòng không ghi riêng).
CLIP_START = 0  #@param {type:"number"}
#@markdown Chữ in ở cạnh dưới clip X (logo đã có sẵn nhờ watermark); để trống = không in.
X_LABEL = "gayvideo.fun"  #@param {type:"string"}
#@markdown Gắn watermark GayVideo.fun trước khi upload (giống luồng hiện tại).
WATERMARK = True  #@param {type:"boolean"}
#@markdown Logo watermark (SVG hoặc PNG): mặc định lấy từ GitHub, nhánh main. Có thể thay bằng URL khác hoặc đường dẫn file, ví dụ trên Drive.
LOGO_URL = "https://raw.githubusercontent.com/ronganminh/stream-video/main/scripts/publish-client/assets/logo-mark.svg"  #@param {type:"string"}
#@markdown Upload thêm lên các host phụ (cần key trong Secrets).
UPLOAD_MIRRORS = False  #@param {type:"boolean"}
#@markdown Lấy thumbnail ở giây thứ (-1 = tự chọn, khoảng 20% thời lượng).
THUMB_AT = -1  #@param {type:"number"}
#@markdown ---
#@markdown **Caption AI cho X.** JoyCaption xem vài khung hình (chỉ dùng nội bộ: không đăng, không gửi lên server), rồi mô hình chữ viết caption kiểu nhá hàng theo giọng văn trong file mẫu, kèm gợi ý tag cho web. Cần GPU: *Runtime → Change runtime type → T4 GPU*. Không có GPU thì dùng caption mặc định. Caption ghi trong LINKS luôn được ưu tiên.
AI_CAPTION = True  #@param {type:"boolean"}
#@markdown Mô hình viết caption: `local` = mô hình mở chạy trên GPU Colab, không cần key. `anthropic` / `openai` = gọi API, cần secret `ANTHROPIC_API_KEY` / `OPENAI_API_KEY`; khi đó mô tả khung hình và tiêu đề được gửi tới nhà cung cấp đó.
AI_TEXT_ENGINE = "local"  #@param ["local", "anthropic", "openai"]
#@markdown Tên mô hình (để trống = mặc định: `Qwen/Qwen3-8B`, `claude-opus-5-5`, `gpt-5-mini`).
AI_TEXT_MODEL = ""  #@param {type:"string"}
#@markdown Số khung hình JoyCaption xem cho mỗi video.
AI_FRAMES = 4  #@param {type:"slider", min:3, max:5, step:1}
#@markdown File caption mẫu (mỗi dòng 1 caption bạn chọn). Chưa có thì notebook tạo từ mẫu khởi đầu.
CAPTION_EXAMPLES = "/content/drive/MyDrive/gayvideo/caption_examples.txt"  #@param {type:"string"}
#@markdown Thêm từ cấm (cách nhau bằng dấu phẩy), ngoài danh sách có sẵn trong `caption_ai.py`.
BANNED_EXTRA = ""  #@param {type:"string"}
#@markdown ---
#@markdown Sổ ghi các video đã đăng (để không đăng trùng). Nên để trên Drive.
MOUNT_DRIVE = True  #@param {type:"boolean"}
LEDGER_PATH = "/content/drive/MyDrive/gayvideo/publish_log.csv"  #@param {type:"string"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif LEDGER_PATH.startswith("/content/drive"):
    LEDGER_PATH = "/content/publish_log.csv"
    print("Cảnh báo: không mount Drive, sổ ghi chỉ tồn tại trong phiên này:", LEDGER_PATH)
if not MOUNT_DRIVE and CAPTION_EXAMPLES.startswith("/content/drive"):
    CAPTION_EXAMPLES = "/content/caption_examples_mine.txt"

In [ ]:
#@title 3. Danh sách video
#@markdown Mỗi dòng: `link | tiêu đề | tag1, tag2 | category | caption X (tiếng Anh) | giây bắt đầu clip | từ khoá tiếng Việt cho AI`. Chỉ link là bắt buộc. Có caption X thì AI không viết đè; từ khoá chỉ để AI hiểu ý, không được đăng. Album → mỗi video một bài, tiêu đề tự thêm "– P1, P2…". Dòng bắt đầu bằng `#` bị bỏ qua.
LINKS = """
# Dòng mẫu (bắt đầu bằng # nên không được đăng). Xoá dấu # và thay bằng nội dung thật:
# https://photos.app.goo.gl/XXXXXXXX | Tiêu đề video | tag1, tag2 | category | English caption | 0 | từ khoá 1, từ khoá 2
"""
#@markdown Hoặc đọc từ CSV (cột: `link,title,tags,category,x_caption,clip_start,keywords`); để trống nếu dùng ô trên.
CSV_PATH = ""  #@param {type:"string"}

from colab_helpers import parse_csv, parse_lines
JOBS = parse_csv(CSV_PATH) if CSV_PATH.strip() else parse_lines(LINKS)
print(f"{len(JOBS)} link")
for job in JOBS:
    print(" -", job.link, "|", job.title or "(tiêu đề tự động)")

### 4. Xác nhận (bắt buộc)
Trước khi đăng, bạn phải xác nhận **cả hai** điều sau cho **mọi** video trong danh sách:
1. Bạn có quyền đăng các video này (bạn là chủ sở hữu, hoặc được chủ sở hữu cho phép).
2. Mọi người xuất hiện trong video đều **từ 18 tuổi trở lên** và đã **đồng ý** quay và đăng video.

Không đánh dấu ô dưới đây thì ô Chuẩn bị và ô Đăng sẽ dừng ngay. Thời điểm xác nhận được ghi vào sổ cùng với từng video.

In [ ]:
#@title Tôi xác nhận có quyền đăng các video này; mọi người trong video đều trên 18 tuổi và đồng ý.
XAC_NHAN = False  #@param {type:"boolean"}
import datetime
XAC_NHAN_LUC = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") if XAC_NHAN else ""
print("Đã xác nhận lúc", XAC_NHAN_LUC) if XAC_NHAN else print("Chưa xác nhận: ô Chuẩn bị và ô Đăng sẽ không chạy.")

In [ ]:
#@title 5. Chuẩn bị: tải video, đề xuất caption (AI)
import html, shutil, tempfile, time
from pathlib import Path

import pandas as pd
from google.colab import userdata
from IPython.display import display

import caption_ai as ai
from colab_helpers import (HostUploader, Ledger, LogoError, SourceError, Step, batch_line, load_logo, probe,
                           redact, register_secret, sha256_file, source_items)
from gayvideo_publish import GayVideoClient, PublishError

if not globals().get("XAC_NHAN"):
    raise SystemExit("Dừng: hãy đánh dấu ô xác nhận ở bước 4 rồi chạy lại ô đó.")

# Kiểm tra logo trước khi đụng tới video nào.
logo = None
if WATERMARK:
    try:
        logo = load_logo(LOGO_URL, Path("/content/gv-logo"))
    except LogoError as error:
        raise SystemExit(f"Dừng: {error} Sửa LOGO_URL ở bước 2 (hoặc tắt WATERMARK) rồi chạy lại.")
    print("Logo OK:", logo.name)


def secret(name):
    try:
        return register_secret(userdata.get(name))
    except Exception:
        return None


client = GayVideoClient(token=secret("GAYVIDEO_API_TOKEN"), base_url=API_URL, timeout=900)
meta = client.meta()
primary = next(host["id"] for host in meta["hosts"] if host["isPrimary"])
targets = [primary] + ([h["id"] for h in meta["hosts"] if not h["isPrimary"]] if UPLOAD_MIRRORS else [])
uploaders = {}
for host in targets:
    key = secret({"dood": "HOST_DOOD_API_KEY", "voe": "HOST_VOE_API_KEY",
                  "earnvids": "HOST_EARNVIDS_API_KEY"}.get(host, ""))
    if key:
        uploaders[host] = HostUploader(host, key)
    elif host == primary:
        raise SystemExit(f"Thiếu key của host chính ({host}) trong Colab Secrets.")
    else:
        print(f"Bỏ qua host phụ {host}: chưa có key trong Secrets.")
categories = {c["slug"] for c in meta["categories"]}
if POST_TO_X and not meta.get("x", {}).get("enabled"):
    print("Lưu ý: server chưa cấu hình Buffer; bài X sẽ chờ trong hàng đợi.")
print(f"Host chính: {primary}; upload lên: {', '.join(uploaders)}")

ledger = Ledger(LEDGER_PATH)
results = []
EXTRA_BANNED = ai.parse_tags(BANNED_EXTRA)
WORK_ROOT = Path("/content/gv-work")
shutil.rmtree(WORK_ROOT, ignore_errors=True)
WORK_ROOT.mkdir(parents=True)
PREPARED, REVIEWED = [], False


def note(job, title, status, detail="", url="", x=""):
    results.append({"link": job.link, "tiêu đề": title, "trạng thái": status,
                    "link web": url, "X": x, "ghi chú": redact(detail, 300)})


# Đọc hết các link trước để biết cả lô có bao nhiêu video.
queue = []
for job in JOBS:
    try:
        with Step(f"Đọc link {job.link[:60]}", indent="") as step:
            album_title, items = source_items(job.link)
            step.extra = f"{len(items)} video"
    except Exception as error:  # one bad link never stops the batch
        print("  ", redact(error, 300))
        note(job, job.title, "lỗi", error)
        ledger.add(link=job.link, status="error", error=redact(error))
        continue
    if not items:
        note(job, job.title, "lỗi", "Không có video nào trong link.")
        continue
    for index, item in enumerate(items):
        base_title = job.title or album_title or f"Video {datetime.date.today():%d-%m-%Y}"
        title = f"{base_title} – P{index + 1}" if len(items) > 1 else base_title
        queue.append((job, item, title))

# Tải từng video về /content/gv-work (giữ tới khi ô Đăng xong).
for number, (job, item, title) in enumerate(queue, 1):
    batch_line(number, len(queue), title)
    if item.key in ledger.done_keys():
        print("  bỏ qua: đã đăng trước đó (theo link)")
        note(job, title, "bỏ qua", "Đã đăng trước đó (theo link).")
        continue
    work = WORK_ROOT / f"{number:03d}"
    work.mkdir()
    digest = ""
    try:
        with Step(f"Tải từ {item.source}") as step:
            source = item.download(work / "source.mp4", step=step)
        with Step("Kiểm tra trùng (sha256)") as step:
            digest = sha256_file(source, step=step)
        if digest in ledger.done_hashes() or digest in {p["digest"] for p in PREPARED}:
            print("  bỏ qua: đã đăng trước đó (cùng file)")
            note(job, title, "bỏ qua", "Đã đăng trước đó (cùng file).")
            ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, status="duplicate")
            shutil.rmtree(work, ignore_errors=True)
            continue
        info = probe(source)
        with Step("Lấy khung hình") as step:
            frames = ai.extract_frames(source, work / "frames", info["duration"], AI_FRAMES if AI_CAPTION else 1)
            step.extra = f"{len(frames)} khung"
        if job.x_caption:
            fitted, problems = ai.check_caption(job.x_caption, EXTRA_BANNED)
            suggestion = ai.Suggestion(fitted, [], "LINKS", "; ".join(problems))
        else:
            suggestion = ai.Suggestion(ai.default_caption(X_CAPTION_TEMPLATE, title, EXTRA_BANNED))
        PREPARED.append({"job": job, "item": item, "title": title, "work": work, "source": source,
                         "digest": digest, "info": info, "frames": frames, "suggestion": suggestion,
                         "tags": list(job.tags)})
    except Exception as error:
        print("  lỗi:", redact(error, 300))
        note(job, title, "lỗi", str(error))
        ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, status="error", error=redact(error))
        shutil.rmtree(work, ignore_errors=True)

# Caption AI: JoyCaption mô tả khung hình, rồi mô hình chữ viết caption + tag.
# Mô tả chỉ nằm trong bộ nhớ của phiên này: không đăng, không ghi sổ, không gửi lên server.
if AI_CAPTION and PREPARED:
    has_gpu, gpu = ai.gpu_status()
    api_key = None
    if AI_TEXT_ENGINE in ("anthropic", "openai"):
        api_key = secret("ANTHROPIC_API_KEY" if AI_TEXT_ENGINE == "anthropic" else "OPENAI_API_KEY")
    if not has_gpu:
        print(f"⚠️ Không chạy được caption AI: {gpu}. Dùng caption trong LINKS hoặc caption mặc định.")
    elif AI_TEXT_ENGINE != "local" and not api_key:
        print(f"⚠️ Thiếu secret {'ANTHROPIC_API_KEY' if AI_TEXT_ENGINE == 'anthropic' else 'OPENAI_API_KEY'}: dùng caption mặc định.")
    else:
        print(f"GPU: {gpu}")
        examples, warnings = ai.load_examples(CAPTION_EXAMPLES, Path("/content/caption_examples.txt").read_text(encoding="utf-8"),
                                              EXTRA_BANNED)
        for warning in warnings:
            print(" ", warning)
        print(f"  {len(examples)} caption mẫu từ {CAPTION_EXAMPLES}")
        descriptions = {}
        captioner = writer = None
        try:
            with Step("Nạp JoyCaption (4-bit)"):
                captioner = ai.JoyCaptioner()
            for number, prepared in enumerate(PREPARED, 1):
                frames = prepared["frames"]
                times = ai.frame_times(prepared["info"]["duration"], len(frames))
                with Step(f"JoyCaption · video {number}/{len(PREPARED)}", unit="khung hình", total=len(frames)) as step:
                    notes = []
                    for second, frame in zip(times, frames):
                        notes.append((second, captioner.describe(frame)))
                        step.update(1)
                descriptions[number] = ai.merge_descriptions(notes)
        except Exception as error:
            print("⚠️ JoyCaption lỗi:", redact(error, 300), "→ dùng caption mặc định.")
        finally:
            if captioner:
                captioner.close()
        if descriptions:
            try:
                with Step(f"Nạp mô hình chữ ({AI_TEXT_ENGINE})"):
                    if AI_TEXT_ENGINE == "anthropic":
                        ai.ensure_package("anthropic")
                        writer = ai.AnthropicWriter(api_key, AI_TEXT_MODEL.strip() or ai.ANTHROPIC_MODEL)
                    elif AI_TEXT_ENGINE == "openai":
                        ai.ensure_package("openai")
                        writer = ai.OpenAIWriter(api_key, AI_TEXT_MODEL.strip() or ai.OPENAI_MODEL)
                    else:
                        writer = ai.LocalWriter(AI_TEXT_MODEL.strip() or ai.LOCAL_TEXT_MODEL)
                for number, prepared in enumerate(PREPARED, 1):
                    if number not in descriptions:
                        continue
                    job = prepared["job"]
                    try:
                        with Step(f"Viết caption · video {number}/{len(PREPARED)}") as step:
                            suggestion = ai.write_caption(
                                writer, description=descriptions[number], title=prepared["title"], tags=job.tags,
                                keywords=job.keywords, examples=examples, extra_banned=EXTRA_BANNED)
                    except Exception as error:
                        prepared["suggestion"].note = f"AI: {redact(error, 200)}; dùng caption mặc định"
                        continue
                    prepared["tags"] = ai.clean_tags(list(job.tags) + suggestion.tags, EXTRA_BANNED)
                    if not job.x_caption:  # caption trong LINKS luôn được ưu tiên
                        prepared["suggestion"] = suggestion
            except Exception as error:
                print("⚠️ Mô hình chữ lỗi:", redact(error, 300), "→ dùng caption mặc định.")
            finally:
                if writer:
                    writer.close()
        descriptions.clear()

if PREPARED:
    display(pd.DataFrame([{"video": p["title"], "nguồn caption": p["suggestion"].source,
                           "caption đề xuất": p["suggestion"].caption, "tag web": ", ".join(p["tags"]),
                           "ghi chú": p["suggestion"].note} for p in PREPARED]))
    print("Tiếp: chạy ô 6 để duyệt caption, rồi ô 7 để đăng.")
else:
    print("Không có video nào để đăng.")

In [ ]:
#@title 6. Duyệt caption (sửa trực tiếp; bỏ chọn "Đăng X" = video vẫn lên web nhưng không đăng X)
import html
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

import caption_ai as ai

if not globals().get("PREPARED"):
    raise SystemExit("Chưa có video nào: chạy ô 5 (Chuẩn bị) trước.")

REVIEW_ROWS, boxes = [], []
for number, prepared in enumerate(PREPARED, 1):
    suggestion = prepared["suggestion"]
    thumb = widgets.Image(value=Path(prepared["frames"][0]).read_bytes(), format="jpg",
                          layout=widgets.Layout(width="180px"))
    post_x = widgets.Checkbox(value=POST_TO_X, description="Đăng X", indent=False, disabled=not POST_TO_X)
    caption = widgets.Textarea(value=suggestion.caption, layout=widgets.Layout(width="100%", height="70px"))
    tags = widgets.Text(value=", ".join(prepared["tags"]), description="Tag web",
                        layout=widgets.Layout(width="100%"))
    status = widgets.HTML()

    def refresh(_=None, post_x=post_x, caption=caption, tags=tags, status=status, suggestion=suggestion):
        decision = ai.review_decision(post_x.value, caption.value, tags.value, EXTRA_BANNED)
        parts = [f"{ai.weight_label(caption.value)} ký tự X", f"nguồn: {suggestion.source}"]
        if suggestion.note:
            parts.append(html.escape(suggestion.note))
        if decision.problems:
            parts.append("<b style='color:#c00'>⚠️ " + html.escape("; ".join(decision.problems)) + " → sẽ không đăng X</b>")
        status.value = " · ".join(parts)

    for control in (post_x, caption, tags):
        control.observe(refresh, "value")
    refresh()
    header = widgets.HTML(f"<b>{number}. {html.escape(prepared['title'])}</b>")
    boxes.append(widgets.HBox([thumb, widgets.VBox([header, post_x, caption, tags, status],
                                                   layout=widgets.Layout(width="75%"))]))
    REVIEW_ROWS.append((prepared, post_x, caption, tags))

display(widgets.VBox(boxes))
REVIEWED = True
print("Sửa xong thì chạy ô 7 (Đăng). Ô Đăng đọc đúng nội dung đang hiện trong các ô trên.")

In [ ]:
#@title 7. Đăng
import shutil, time
from pathlib import Path

import pandas as pd
from IPython.display import display

import caption_ai as ai
from colab_helpers import Step, batch_line, cut_x_clip, make_thumbnail, redact, watermark
from gayvideo_publish import PublishError

if not globals().get("XAC_NHAN"):
    raise SystemExit("Dừng: hãy đánh dấu ô xác nhận ở bước 4 rồi chạy lại ô đó.")
if not globals().get("PREPARED"):
    raise SystemExit("Chưa có video nào: chạy ô 5 (Chuẩn bị) trước.")
if AI_CAPTION and POST_TO_X and not globals().get("REVIEWED"):
    raise SystemExit("Hãy duyệt caption ở ô 6 trước khi đăng.")

# Caption đã duyệt (hoặc caption đề xuất nếu không dùng AI và chưa mở ô duyệt).
decisions = {}
for prepared in PREPARED:
    decisions[id(prepared)] = ai.review_decision(POST_TO_X, prepared["suggestion"].caption,
                                                 ", ".join(prepared["tags"]), EXTRA_BANNED)
if globals().get("REVIEWED"):
    for prepared, post_x, caption, tags in REVIEW_ROWS:
        decisions[id(prepared)] = ai.review_decision(POST_TO_X and post_x.value, caption.value, tags.value,
                                                     EXTRA_BANNED)

clip_seconds = client.clip_seconds() if POST_TO_X else 0
batch_started = time.monotonic()
for number, prepared in enumerate(PREPARED, 1):
    job, item, title, work = prepared["job"], prepared["item"], prepared["title"], prepared["work"]
    source, info, digest = prepared["source"], prepared["info"], prepared["digest"]
    decision = decisions[id(prepared)]
    batch_line(number, len(PREPARED), title)
    if POST_TO_X and decision.problems:
        print("  không đăng X:", "; ".join(decision.problems))
    codes = {}
    video_started = time.monotonic()
    try:
        if item.key in ledger.done_keys() or digest in ledger.done_hashes():
            print("  bỏ qua: đã đăng trước đó")
            note(job, title, "bỏ qua", "Đã đăng trước đó.")
            continue
        upload_file = source
        if WATERMARK:
            with Step("Watermark (ffmpeg)") as step:
                upload_file = watermark(source, work / "upload.mp4", logo, info["width"], info["duration"], step)

        warnings = []
        for host, uploader in uploaders.items():
            try:
                with Step(f"Upload lên {host}") as step:
                    codes[host] = uploader.upload(upload_file, title, decision.tags or job.tags, step=step)
                    step.extra = f"mã {codes[host]}"
            except Exception as error:
                if host == primary:
                    raise
                warnings.append(f"{host}: {redact(error, 150)}")

        with Step("Thumbnail") as step:
            thumb = make_thumbnail(source, work / "thumb.jpg", info["duration"], None if THUMB_AT < 0 else THUMB_AT)
        category = job.category if job.category in categories else None
        if job.category and not category:
            warnings.append(f"category '{job.category}' không tồn tại, đã bỏ")
        if POST_TO_X and decision.problems:
            warnings.append("không đăng X: " + "; ".join(decision.problems))

        caption, clip_id = None, None
        if decision.post_x:
            caption = decision.caption
            with Step("Cắt clip X (ffmpeg)") as step:
                clip = cut_x_clip(upload_file, work / "clip.mp4", info["duration"],
                                  start=job.clip_start if job.clip_start is not None else CLIP_START,
                                  seconds=clip_seconds, label=X_LABEL.strip() or None, step=step)
            with Step("Upload clip X lên web") as step:
                step.total = clip.stat().st_size
                clip_id = client.upload_clip(str(clip), progress=step.set)["clipId"]
        with Step("Publish lên web") as step:
            result = client.publish(
                title, codes, tags=decision.tags or None, category=category, thumbnail=str(thumb), publish=PUBLISH,
                x_caption=caption, x_clip_id=clip_id,
            )
        video = result["video"]
        url = API_URL.rstrip("/") + video["url"]
        x_status = (result.get("x") or {}).get("status", "")
        ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, video_id=video["id"],
                   url=url, x=x_status, status="done", error=f"xác nhận {XAC_NHAN_LUC}")
        note(job, title, "đã đăng" if video["published"] else "bản nháp", "; ".join(warnings + result["warnings"]),
             url, x_status)
        print(f"  xong video {number}/{len(PREPARED)} sau {time.monotonic() - video_started:.0f} s: {url}")
    except Exception as error:
        detail = f"{error.code}: {error}" if isinstance(error, PublishError) else str(error)
        if codes:
            detail += f" (đã upload: {codes}; có thể đăng lại thủ công bằng mã file này)"
        print("  lỗi:", redact(detail, 300))
        note(job, title, "lỗi", detail)
        ledger.add(link=job.link, item_key=item.key, sha256=digest, title=title, status="error", error=redact(detail))
    finally:
        shutil.rmtree(work, ignore_errors=True)

print(f"Cả lô: {len(PREPARED)} video trong {(time.monotonic() - batch_started) / 60:.1f} phút.")
PREPARED, REVIEW_ROWS, REVIEWED = [], [], False
shutil.rmtree(WORK_ROOT, ignore_errors=True)

pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(results))
print("Sổ ghi:", LEDGER_PATH)

### Gặp lỗi?
- **"Google Photos …"**: link chưa chia sẻ công khai, hoặc Google chặn tải tự động. Dùng link Google Drive thay thế (xem đầu notebook).
- **"Thiếu key …"**: thêm secret tương ứng trong 🔑 Secrets và bật *Notebook access*.
- **`401 unauthorized`**: token sai hoặc đã bị thu hồi. Tạo token mới trên VPS: `npm run api-token -- create <email> colab`.
- **Lỗi sau khi đã upload host:** bảng kết quả có ghi mã file đã upload. Chạy lại notebook sẽ upload lại; muốn tránh thì đăng thủ công bằng mã file đó (xem `docs/API.md`).
- **"Không chạy được caption AI"**: runtime chưa có GPU, hoặc thiếu secret API. Notebook vẫn chạy với caption trong LINKS hoặc caption mặc định.
- **Hết bộ nhớ GPU (CUDA out of memory)**: *Runtime → Restart session*, chạy lại từ ô 1; hoặc giảm `AI_FRAMES`, hoặc đổi `AI_TEXT_MODEL` sang mô hình nhỏ hơn (ví dụ `Qwen/Qwen3-4B-Instruct-2507`).
- **Xem bài X:** `GET /api/v1/videos/<id>` (trường `social`), hoặc trên VPS chạy `npm run social:status`.